# IM 分钟混合贡献的日内季节性

本文使用交易日和日内分钟两个索引；符号在首次使用时定义，代码字段保持现有名称。收益及其周期尺度沿用 Boudt 等收益过滤文献的符号约定，其他文献须按定义映射，不能仅凭相同字母认定是同一对象。完整依据与方法边界见[项目符号与方法说明](../volatility_research/README.md)。

| 符号 | 含义 | 代码对应或用途 |
|---|---|---|
| $t$；$i=1,\ldots,M$ | 交易日序号；全天分钟位置，$M=240$ | 09:31 为 $i=1$，13:01 为 $i=121$ |
| $L$ | 回望交易日数，取 30、60、120、360 | `lookback_windows` |
| $r_{t,i}$ | 分钟对数收益率 | `log_return` |
| $f_{t,i}$；$f_{t,i}^{2}$ | 收益的标准差周期；相应方差周期，约定 $M^{-1}\sum_i f_{t,i}^{2}=1$ | 用于区分统计对象；本 Notebook 未直接估计 $f$ |
| $\widehat f_{t,i}$；$r^f_{t,i}=r_{t,i}/\widehat f_{t,i}$ | 估计的标准差周期；据此过滤的收益 | 帽号表示估计量；不要与下文旧收益调整构造混用 |
| $RV_t$；$BV_t$（亦称 BPV） | 收益平方和；绝对收益相邻乘积构成的实现双幂变差 | 沿用文献常见名称；本项目只在同一 Session 内配对 |
| $b_{t,i}$ | $\frac{\pi}{2}\lvert r_{t,i}\rvert\lvert r_{t,i-1}\rvert$，仅限有效同 Session 配对 | `bpv_contribution`；$b$ 是本项目对 BV 求和项的简称 |
| $c^{\mathrm{mix}}_{t,i}$ | 首分钟收益平方、其余分钟 $b_{t,i}$ 构成的混合贡献 | `variance_contribution` |
| $x^{\mathrm{mix}}_{t,i}$ | 混合贡献除以当日平均混合贡献 | `daily_shape_matrix` |
| $\widehat g^{\mathrm{mix},(L)}_{t,i}$ | 过去 $L$ 个交易日估计的混合贡献季节因子 | `seasonality_by_lookback[L]`；输出字段 `variance_seasonality` |

$c$、$x$、$g$ 及上标 $\mathrm{mix}$ 是本项目为现有混合构造明确约定的记号，并非文献统一命名。$\widehat g^{\mathrm{mix},(L)}$ 调整的是贡献形状，不能由其构造直接断言它就是 $f^2$。

#### Cell 1：导入依赖，设置路径和滚动窗口。

In [1]:
import pathlib
import sys

project_markers = [".git", ".env", "config/settings.py"]
current_path = pathlib.Path.cwd().resolve()

for candidate_root in [current_path, *current_path.parents]:
    if all((candidate_root / marker).exists() for marker in project_markers):
        sys.path.insert(0, str(candidate_root))
        sys.path.insert(0, str(candidate_root / "02_Futures_Lakehouse"))
        break
else:
    raise RuntimeError("未找到项目根目录")

from config.settings import settings
from config.data_contracts import TRADE_CALENDAR_SCHEMA, arrow_to_pandas
from a00_03_notebook_schema_browser import display_schema_metadata

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.parquet as pq

research_dir = (
    candidate_root / "01_project_collection/JQ_strategy/draft_experiments_v2"
)
source_path = research_dir / "im_main_continuous_1m.parquet"
lookback_windows = (30, 60, 120, 360)  # 单位：交易日
as_of_at = pd.Timestamp.now(tz="Asia/Shanghai")

#### Cell 2：读取实际合约的原始收盘价，按交易日和时间排序。

In [11]:
minute_input_schema = pa.schema([
    pa.field("contract_code", pa.string(), nullable=False),
    pa.field("trading_date", pa.date32(), nullable=False),
    pa.field("session_number", pa.int8(), nullable=False),
    pa.field("bar_at", pa.timestamp("us", tz="Asia/Shanghai"), nullable=False),
    pa.field("open", pa.float64(), nullable=False),
    pa.field("close", pa.float64(), nullable=False),
])

minute_table = pq.read_table(source_path, columns=minute_input_schema.names)
minute_df = (
    minute_table.cast(minute_input_schema)
    .to_pandas(types_mapper=pd.ArrowDtype)
    .sort_values(["trading_date", "bar_at"])
    .reset_index(drop=True)
)
minute_df[["open", "close"]] = minute_df[["open", "close"]].astype("float64")

#### Cell 3：读取完整交易日序列，建立每天 240 个分钟位置，供后续矩阵对齐。

In [12]:
display_schema_metadata([TRADE_CALENDAR_SCHEMA])

calendar_table_name = (
    TRADE_CALENDAR_SCHEMA.metadata[b"table_name"].decode("utf-8")
)
calendar_primary_key = (
    TRADE_CALENDAR_SCHEMA.metadata[b"primary_key"].decode("utf-8").split(",")
)
calendar_partition_columns = (
    TRADE_CALENDAR_SCHEMA.metadata[b"partition_columns"]
    .decode("utf-8").split(",")
)
calendar_read_schema = pa.schema([
    TRADE_CALENDAR_SCHEMA.field("calendar_date"),
])
calendar_partitioning = ds.partitioning(
    pa.schema([
        TRADE_CALENDAR_SCHEMA.field(column)
        for column in calendar_partition_columns
    ]),
    flavor="hive",
)

calendar_dataset = ds.dataset(
    settings.futures_lake_root / "silver" / calendar_table_name,
    format="parquet",
    partitioning=calendar_partitioning,
)
calendar_table = calendar_dataset.to_table(
    columns=calendar_read_schema.names,
    filter=(
        (ds.field("calendar_date") >= minute_df["trading_date"].min())
        & (ds.field("calendar_date") <= minute_df["trading_date"].max())
        & ds.field("is_trading_day")
    ),
)
calendar_df = arrow_to_pandas(calendar_table, calendar_read_schema)
calendar_df = calendar_df.sort_values(calendar_primary_key)

trading_dates = pd.Index(
    calendar_df["calendar_date"].tolist(),
    name="trading_date",
)
minute_slots = pd.Index([
    *pd.date_range("2000-01-01 09:31", "2000-01-01 11:30", freq="min").time,
    *pd.date_range("2000-01-01 13:01", "2000-01-01 15:00", freq="min").time,
], name="minute_slot")

英文表名,中文表名,字段数,主键,Hive 分区,Schema 版本
dim_trade_calendar,中国期货交易日历维度表,11,calendar_date,year,1.1.0


### Session 首分钟的收益起点

收益按“交易日＋合约＋Session”分组计算。09:31 和 13:01
分别对应全天分钟位置 $i=1$ 和 $i=121$，是各 Session 的第一根分钟线，
没有组内前一分钟的 close。因此直接 diff() 会产生空值。
这属于计算边界，不代表源数据漏采。

记 $P_{t,i}$、$P^{\mathrm{open}}_{t,i}$ 分别为第 $t$ 个交易日、第 $i$ 根分钟线的原始收盘价和开盘价。
首分钟使用本根分钟线的原始 open 作为收益起点：

$$
r_{t,i}=\log(P_{t,i}/P^{\mathrm{open}}_{t,i}),\qquad i\in\{1,121\}.
$$

其余分钟使用同一 Session 内相邻收盘价的对数差：

$$
r_{t,i}=\log(P_{t,i}/P_{t,i-1}).
$$

这样不会把隔夜或午休期间的价格变化混入一分钟收益。
open 仅用于 Session 首分钟；盘中时间缺口仍保留为空，不套用这一处理。

#### Cell 4：在同一交易日、合约和 Session 内计算一分钟收益

In [14]:
import datetime as dt

minute_df["minute_slot"] = minute_df["bar_at"].dt.time
minute_df["log_close"] = np.log(
    minute_df["close"].where(minute_df["close"] > 0)
)

minute_groups = minute_df.groupby(
    ["trading_date", "contract_code", "session_number"],
    sort=False,
)
minute_interval = minute_groups["bar_at"].diff()

minute_df["log_return"] = (
    minute_groups["log_close"].diff()
    .where(minute_interval.eq(pd.Timedelta(minutes=1)))
)

is_session_open_minute = minute_df["minute_slot"].isin([
    dt.time(9, 31), dt.time(13, 1),
])
session_open_prices = minute_df.loc[is_session_open_minute, "open"]

minute_df.loc[is_session_open_minute, "log_return"] = (
    minute_df.loc[is_session_open_minute, "log_close"]
    - np.log(session_open_prices.where(session_open_prices > 0))
)

minute_df.head()

,contract_code,trading_date,session_number,bar_at,open,close,minute_slot,log_close,log_return
0,IM2208.CCFX,2022-07-25,1,2022-07-25 09:31:00+08:00,6974.2,6973.8,09:31:00,8.849916,-0.000057
1,IM2208.CCFX,2022-07-25,1,2022-07-25 09:32:00+08:00,6974.0,6982.8,09:32:00,8.851205,0.001290
2,IM2208.CCFX,2022-07-25,1,2022-07-25 09:33:00+08:00,6982.4,6989.8,09:33:00,8.852207,0.001002
3,IM2208.CCFX,2022-07-25,1,2022-07-25 09:34:00+08:00,6989.6,6993.0,09:34:00,8.852665,0.000458
4,IM2208.CCFX,2022-07-25,1,2022-07-25 09:35:00+08:00,6993.6,7002.2,09:35:00,8.853980,0.001315


### 首分钟 RV 项与其余分钟 BV 求和项

实现方差的常见记号是 $RV_t=\sum_{i=1}^{M}r_{t,i}^{2}$。
实现双幂变差通常记作 $BV$，也常简称 BPV。对有效的同 Session 相邻收益，
本项目用 $b_{t,i}$ 简称 BV 的一个求和项：

$$
b_{t,i}=\frac{\pi}{2}|r_{t,i}||r_{t,i-1}|.
$$

补入首分钟收益后，仍缺少与它配对的前一个组内收益。
因此现有代码构造的是下面的**混合贡献**，而非每个分钟均为 BV 求和项：

$$
c^{\mathrm{mix}}_{t,i}
=
\begin{cases}
r_{t,i}^{2}, & i\in\{1,121\},\\
b_{t,i}, & \text{其余有效同 Session 配对分钟}.
\end{cases}
$$

所以 09:31、13:01 使用 RV 的平方收益项，09:32、13:02 起使用 BV 求和项。
$c^{\mathrm{mix}}$ 是本项目定义；其每日合计不能直接称为普通 BV。
代码将混合贡献存入 `variance_contribution`，通过 `contribution_method`
标明类型。缺失收益或无效配对仍保留为空。

#### Cell 5：计算 BV 求和项与首分钟 RV 项，整理混合贡献矩阵。

`bpv_contribution` 保存有效相邻绝对收益乘积；
`variance_contribution_matrix` 保存“交易日 × 分钟位置”的混合贡献 $c^{\mathrm{mix}}_{t,i}$。

In [15]:
previous_log_return = minute_groups["log_return"].shift(1)

minute_df["bpv_contribution"] = (
    (np.pi / 2)
    * minute_df["log_return"].abs()
    * previous_log_return.abs()
)

minute_df["variance_contribution"] = minute_df["bpv_contribution"]
minute_df.loc[is_session_open_minute, "variance_contribution"] = (
    minute_df.loc[is_session_open_minute, "log_return"].pow(2)
)
minute_df["contribution_method"] = np.where(
    is_session_open_minute, "rv_open", "bpv"
)

variance_contribution_matrix = minute_df.pivot(
    index="trading_date",
    columns="minute_slot",
    values="variance_contribution",
).reindex(
    index=trading_dates,
    columns=minute_slots,
)

In [16]:
variance_contribution_matrix.head()

minute_slot,09:31:00,09:32:00,09:33:00,09:34:00,09:35:00,09:36:00,09:37:00,09:38:00,09:39:00,09:40:00,09:41:00,09:42:00,09:43:00,09:44:00,09:45:00,09:46:00,09:47:00,09:48:00,09:49:00,09:50:00,09:51:00,09:52:00,09:53:00,09:54:00,09:55:00,09:56:00,09:57:00,09:58:00,09:59:00,10:00:00,10:01:00,10:02:00,10:03:00,10:04:00,10:05:00,10:06:00,10:07:00,10:08:00,10:09:00,10:10:00,10:11:00,10:12:00,10:13:00,10:14:00,10:15:00,10:16:00,10:17:00,10:18:00,10:19:00,10:20:00,10:21:00,10:22:00,10:23:00,10:24:00,10:25:00,10:26:00,10:27:00,10:28:00,10:29:00,10:30:00,10:31:00,10:32:00,10:33:00,10:34:00,10:35:00,10:36:00,10:37:00,10:38:00,10:39:00,10:40:00,10:41:00,10:42:00,10:43:00,10:44:00,10:45:00,10:46:00,10:47:00,10:48:00,10:49:00,10:50:00,10:51:00,10:52:00,10:53:00,10:54:00,10:55:00,10:56:00,10:57:00,10:58:00,10:59:00,11:00:00,11:01:00,11:02:00,11:03:00,11:04:00,11:05:00,11:06:00,11:07:00,11:08:00,11:09:00,11:10:00,11:11:00,11:12:00,11:13:00,11:14:00,11:15:00,11:16:00,11:17:00,11:18:00,11:19:00,11:20:00,11:21:00,11:22:00,11:23:00,11:24:00,11:25:00,11:26:00,11:27:00,11:28:00,11:29:00,11:30:00,13:01:00,13:02:00,13:03:00,13:04:00,13:05:00,13:06:00,13:07:00,13:08:00,13:09:00,13:10:00,13:11:00,13:12:00,13:13:00,13:14:00,13:15:00,13:16:00,13:17:00,13:18:00,13:19:00,13:20:00,13:21:00,13:22:00,13:23:00,13:24:00,13:25:00,13:26:00,13:27:00,13:28:00,13:29:00,13:30:00,13:31:00,13:32:00,13:33:00,13:34:00,13:35:00,13:36:00,13:37:00,13:38:00,13:39:00,13:40:00,13:41:00,13:42:00,13:43:00,13:44:00,13:45:00,13:46:00,13:47:00,13:48:00,13:49:00,13:50:00,13:51:00,13:52:00,13:53:00,13:54:00,13:55:00,13:56:00,13:57:00,13:58:00,13:59:00,14:00:00,14:01:00,14:02:00,14:03:00,14:04:00,14:05:00,14:06:00,14:07:00,14:08:00,14:09:00,14:10:00,14:11:00,14:12:00,14:13:00,14:14:00,14:15:00,14:16:00,14:17:00,14:18:00,14:19:00,14:20:00,14:21:00,14:22:00,14:23:00,14:24:00,14:25:00,14:26:00,14:27:00,14:28:00,14:29:00,14:30:00,14:31:00,14:32:00,14:33:00,14:34:00,14:35:00,14:36:00,14:37:00,14:38:00,14:39:00,14:40:00,14:41:00,14:42:00,14:43:00,14:44:00,14:45:00,14:46:00,14:47:00,14:48:00,14:49:00,14:50:00,14:51:00,14:52:00,14:53:00,14:54:00,14:55:00,14:56:00,14:57:00,14:58:00,14:59:00,15:00:00
trading_date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2022-07-25,3.289698e-09,1.161959e-07,2.029849e-06,7.203716e-07,9.452452e-07,1.709906e-06,7.425461e-07,4.096514e-07,5.936471e-07,1.111866e-07,2.992373e-07,3.700619e-06,1.758308e-06,1.520590e-06,2.095229e-06,2.028622e-06,1.472756e-06,1.428002e-06,5.409673e-07,8.661677e-07,3.471532e-06,2.610980e-06,3.313059e-06,9.190267e-07,9.055102e-08,2.809299e-07,3.615068e-07,4.201099e-07,2.384528e-06,2.984648e-06,5.263390e-07,4.911014e-07,1.638064e-06,1.675596e-06,1.170610e-06,2.719066e-08,7.113807e-08,1.847389e-06,7.727661e-07,1.578082e-06,1.925480e-06,0.000000e+00,0.000000e+00,1.169995e-07,3.315321e-07,3.755419e-07,7.950936e-07,1.967092e-06,4.379378e-07,1.460718e-07,2.555005e-07,8.202235e-07,2.163341e-06,1.774763e-06,6.226884e-07,3.700718e-07,4.269079e-07,1.940992e-07,1.553841e-07,6.370110e-07,2.812779e-06,2.127774e-06,7.628962e-07,1.230513e-07,8.418816e-08,1.177991e-07,2.446598e-07,3.147342e-07,2.915629e-07,0.000000e+00,0.000000e+00,9.328806e-08,7.770879e-09,3.366801e-08,4.882212e-07,1.315953e-06,1.773244e-06,8.121312e-07,4.167833e-07,2.083437e-07,3.956138e-07,2.320390e-06,1.768203e-06,8.280623e-07,3.716595e-07,1.014229e-07,2.729283e-07,1.818580e-06,1.766900e-06,1.632857e-06,4.795003e-08,6.342122e-08,1.266987e-06,1.241825e-06,1.493188e-06,5.920310e-07,2.962284e-07,4.366915e-07,6.191894e-07,4.426602e-08,8.598509e-08,3.442972e-07,4.176487e-08,6.789630e-08,1.528129e-07,3.174814e-07,2.117734e-07,2.353648e-08,3.530014e-08,3.295298e-07,7.329968e-07,7.596027e-07,4.554328e-07,4.081600e-07,1.701526e-07,1.962902e-07,2.001756e-06,4.278810e-06,8.508109e-06,2.669822e-06,3.24

#### Cell 6：计算每日混合贡献总量、有效贡献数及历史样本资格。

`daily_variance_proxy` 是 $\sum_i c^{\mathrm{mix}}_{t,i}$，包含两个首分钟 RV 项。
只有早于运行当天、240 个贡献均有限且日总量为正的完整交易日，才能用于历史形状估计。

In [18]:
daily_summary_df = pd.DataFrame({
    "daily_variance_proxy": variance_contribution_matrix.sum(
        axis=1, min_count=1
    ),
    "valid_slot_count": variance_contribution_matrix.count(axis=1),
})

daily_summary_df["is_eligible_history_day"] = (
    (trading_dates < as_of_at.date())
    & np.isfinite(variance_contribution_matrix).all(axis=1)
    & np.isfinite(daily_summary_df["daily_variance_proxy"])
    & daily_summary_df["daily_variance_proxy"].gt(0)
)

daily_summary_df.head()

,daily_variance_proxy,valid_slot_count,is_eligible_history_day
trading_date,,,
2022-07-25,0.000187,240,True
2022-07-26,0.000137,240,True
2022-07-27,0.000085,240,True
2022-07-28,0.000084,240,True
2022-07-29,0.000125,240,True


### 每日归一化：提取混合贡献的日内相对形状

将每分钟混合贡献除以当天平均分钟混合贡献。对合格的完整历史日：

$$
x^{\mathrm{mix}}_{t,i}
=
\frac{c^{\mathrm{mix}}_{t,i}}
     {M^{-1}\sum_{j=1}^{M}c^{\mathrm{mix}}_{t,j}}
=
\frac{M c^{\mathrm{mix}}_{t,i}}
     {\sum_{j=1}^{M}c^{\mathrm{mix}}_{t,j}},
\qquad M=240.
$$

$x^{\mathrm{mix}}$ 是本项目定义的“日内相对混合贡献”。
其日内均值为 1；数值 2 表示该分钟贡献是当天平均分钟贡献的两倍。
这一步消去当天混合贡献的总体水平，保留各分钟的相对形状。

代码先使用 `valid_slot_count` 计算，再将不符合完整历史日条件的整行置空；
因此进入滚动估计的有效日满足上式中的 $M=240$。
历史日的全日归一化在估计目标日因子前已经完成；目标日仅使用此前交易日的形状。

#### Cell 7：用“有效贡献数 × 混合贡献 ÷ 日总量”计算 $x^{\mathrm{mix}}_{t,i}$，不合格历史日置空。

In [21]:
daily_shape_matrix = (
    variance_contribution_matrix
    .mul(daily_summary_df["valid_slot_count"], axis=0)
    .div(
        daily_summary_df["daily_variance_proxy"].where(
            daily_summary_df["daily_variance_proxy"] > 0
        ),
        axis=0,
    )
)

daily_shape_matrix.loc[
    ~daily_summary_df["is_eligible_history_day"], :
] = np.nan

daily_shape_matrix.head()

minute_slot,09:31:00,09:32:00,09:33:00,09:34:00,09:35:00,09:36:00,09:37:00,09:38:00,09:39:00,09:40:00,09:41:00,09:42:00,09:43:00,09:44:00,09:45:00,09:46:00,09:47:00,09:48:00,09:49:00,09:50:00,09:51:00,09:52:00,09:53:00,09:54:00,09:55:00,09:56:00,09:57:00,09:58:00,09:59:00,10:00:00,10:01:00,10:02:00,10:03:00,10:04:00,10:05:00,10:06:00,10:07:00,10:08:00,10:09:00,10:10:00,10:11:00,10:12:00,10:13:00,10:14:00,10:15:00,10:16:00,10:17:00,10:18:00,10:19:00,10:20:00,10:21:00,10:22:00,10:23:00,10:24:00,10:25:00,10:26:00,10:27:00,10:28:00,10:29:00,10:30:00,10:31:00,10:32:00,10:33:00,10:34:00,10:35:00,10:36:00,10:37:00,10:38:00,10:39:00,10:40:00,10:41:00,10:42:00,10:43:00,10:44:00,10:45:00,10:46:00,10:47:00,10:48:00,10:49:00,10:50:00,10:51:00,10:52:00,10:53:00,10:54:00,10:55:00,10:56:00,10:57:00,10:58:00,10:59:00,11:00:00,11:01:00,11:02:00,11:03:00,11:04:00,11:05:00,11:06:00,11:07:00,11:08:00,11:09:00,11:10:00,11:11:00,11:12:00,11:13:00,11:14:00,11:15:00,11:16:00,11:17:00,11:18:00,11:19:00,11:20:00,11:21:00,11:22:00,11:23:00,11:24:00,11:25:00,11:26:00,11:27:00,11:28:00,11:29:00,11:30:00,13:01:00,13:02:00,13:03:00,13:04:00,13:05:00,13:06:00,13:07:00,13:08:00,13:09:00,13:10:00,13:11:00,13:12:00,13:13:00,13:14:00,13:15:00,13:16:00,13:17:00,13:18:00,13:19:00,13:20:00,13:21:00,13:22:00,13:23:00,13:24:00,13:25:00,13:26:00,13:27:00,13:28:00,13:29:00,13:30:00,13:31:00,13:32:00,13:33:00,13:34:00,13:35:00,13:36:00,13:37:00,13:38:00,13:39:00,13:40:00,13:41:00,13:42:00,13:43:00,13:44:00,13:45:00,13:46:00,13:47:00,13:48:00,13:49:00,13:50:00,13:51:00,13:52:00,13:53:00,13:54:00,13:55:00,13:56:00,13:57:00,13:58:00,13:59:00,14:00:00,14:01:00,14:02:00,14:03:00,14:04:00,14:05:00,14:06:00,14:07:00,14:08:00,14:09:00,14:10:00,14:11:00,14:12:00,14:13:00,14:14:00,14:15:00,14:16:00,14:17:00,14:18:00,14:19:00,14:20:00,14:21:00,14:22:00,14:23:00,14:24:00,14:25:00,14:26:00,14:27:00,14:28:00,14:29:00,14:30:00,14:31:00,14:32:00,14:33:00,14:34:00,14:35:00,14:36:00,14:37:00,14:38:00,14:39:00,14:40:00,14:41:00,14:42:00,14:43:00,14:44:00,14:45:00,14:46:00,14:47:00,14:48:00,14:49:00,14:50:00,14:51:00,14:52:00,14:53:00,14:54:00,14:55:00,14:56:00,14:57:00,14:58:00,14:59:00,15:00:00
trading_date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2022-07-25,0.004215,0.148880,2.600817,0.923002,1.211130,2.190879,0.951414,0.524881,0.760632,0.142462,0.383409,4.741552,2.252895,1.948311,2.684588,2.599246,1.887022,1.829679,0.693134,1.109809,4.448025,3.345412,4.244977,1.177536,0.116022,0.359952,0.463194,0.538281,3.055262,3.824188,0.674391,0.629241,2.098829,2.146918,1.499886,0.034839,0.091148,2.367034,0.990134,2.021975,2.467091,0.000000,0.000000,0.149910,0.424787,0.481177,1.018742,2.520408,0.561124,0.187160,0.327369,1.050941,2.771859,2.273980,0.797842,0.474168,0.546991,0.248697,0.199092,0.816193,3.603975,2.726287,0.977488,0.157664,0.107869,0.150934,0.313479,0.403265,0.373575,0.000000,0.000000,0.119529,0.009957,0.043138,0.625551,1.686112,2.272032,1.040572,0.534019,0.266948,0.506894,2.973083,2.265574,1.060985,0.476202,0.129952,0.349699,2.330121,2.263905,2.092156,0.061438,0.081261,1.623373,1.591133,1.913202,0.758561,0.379553,0.559527,0.793359,0.056717,0.110171,0.441143,0.053513,0.086995,0.195797,0.406785,0.271342,0.030157,0.045230,0.422222,0.939179,0.973268,0.583540,0.522970,0.218014,0.251504,2.564823,5.482380,10.901321,3.420805,4.161619,0.316608,0.234978,0.312917,0.020397,0.173316,1.963366,0.288852,0.076477,0.550166,2.443727,2.036853,1.884554,0.062860,0.032306,1.326106,1.328423,0.162201,0.460825,0.828350,0.337407,0.225092,0.061403,0.102362,0.734333,0.955788,0.088927,0.006838,0.027345,0.027331,0.119665,3.595648,2.618121,0.087104,0.046063,0.092058,0.241747,0.120626,0.018672,0.037356,0.047564,0.023793,0.018702,0.187051,0.407915,2.080157,1.388668,0.136298,0.076685,0.245494,0.846478,0.900182,0.493749,0.3775

#### Cell 8：先滞后一个交易日，再计算过去 $L$ 个交易日的逐分钟历史中位数。

`shift(1)` 保证目标日 $t$ 仅使用截至 $t-1$ 的历史。
滚动窗口按完整交易日序列计数，并要求窗口内 $L$ 个有效观测；
遇到缺失历史日时，不向更早日期补足。下方既有打印标签“回望日历天数”应按 $L$ 个交易日理解。

In [34]:
lagged_shape_matrix = daily_shape_matrix.shift(1)

rolling_median_by_lookback = {}

for lookback in lookback_windows:
    rolling_median_by_lookback[lookback] = (
        lagged_shape_matrix
        .rolling(window=lookback, min_periods=lookback)
        .median()
    )

print(rolling_median_by_lookback.keys())
for key, value in  rolling_median_by_lookback.items():
    print("回望日历天数 lookback: ", key)
    print(value.columns)
    display(value.tail())

dict_keys([30, 60, 120, 360])
回望日历天数 lookback:  30
Index([09:31:00, 09:32:00, 09:33:00, 09:34:00, 09:35:00, 09:36:00, 09:37:00, 09:38:00, 09:39:00, 09:40:00,
       ...
       14:51:00, 14:52:00, 14:53:00, 14:54:00, 14:55:00, 14:56:00, 14:57:00, 14:58:00, 14:59:00, 15:00:00], dtype='object', name='minute_slot', length=240)


minute_slot,09:31:00,09:32:00,09:33:00,09:34:00,09:35:00,09:36:00,09:37:00,09:38:00,09:39:00,09:40:00,09:41:00,09:42:00,09:43:00,09:44:00,09:45:00,09:46:00,09:47:00,09:48:00,09:49:00,09:50:00,09:51:00,09:52:00,09:53:00,09:54:00,09:55:00,09:56:00,09:57:00,09:58:00,09:59:00,10:00:00,10:01:00,10:02:00,10:03:00,10:04:00,10:05:00,10:06:00,10:07:00,10:08:00,10:09:00,10:10:00,10:11:00,10:12:00,10:13:00,10:14:00,10:15:00,10:16:00,10:17:00,10:18:00,10:19:00,10:20:00,10:21:00,10:22:00,10:23:00,10:24:00,10:25:00,10:26:00,10:27:00,10:28:00,10:29:00,10:30:00,10:31:00,10:32:00,10:33:00,10:34:00,10:35:00,10:36:00,10:37:00,10:38:00,10:39:00,10:40:00,10:41:00,10:42:00,10:43:00,10:44:00,10:45:00,10:46:00,10:47:00,10:48:00,10:49:00,10:50:00,10:51:00,10:52:00,10:53:00,10:54:00,10:55:00,10:56:00,10:57:00,10:58:00,10:59:00,11:00:00,11:01:00,11:02:00,11:03:00,11:04:00,11:05:00,11:06:00,11:07:00,11:08:00,11:09:00,11:10:00,11:11:00,11:12:00,11:13:00,11:14:00,11:15:00,11:16:00,11:17:00,11:18:00,11:19:00,11:20:00,11:21:00,11:22:00,11:23:00,11:24:00,11:25:00,11:26:00,11:27:00,11:28:00,11:29:00,11:30:00,13:01:00,13:02:00,13:03:00,13:04:00,13:05:00,13:06:00,13:07:00,13:08:00,13:09:00,13:10:00,13:11:00,13:12:00,13:13:00,13:14:00,13:15:00,13:16:00,13:17:00,13:18:00,13:19:00,13:20:00,13:21:00,13:22:00,13:23:00,13:24:00,13:25:00,13:26:00,13:27:00,13:28:00,13:29:00,13:30:00,13:31:00,13:32:00,13:33:00,13:34:00,13:35:00,13:36:00,13:37:00,13:38:00,13:39:00,13:40:00,13:41:00,13:42:00,13:43:00,13:44:00,13:45:00,13:46:00,13:47:00,13:48:00,13:49:00,13:50:00,13:51:00,13:52:00,13:53:00,13:54:00,13:55:00,13:56:00,13:57:00,13:58:00,13:59:00,14:00:00,14:01:00,14:02:00,14:03:00,14:04:00,14:05:00,14:06:00,14:07:00,14:08:00,14:09:00,14:10:00,14:11:00,14:12:00,14:13:00,14:14:00,14:15:00,14:16:00,14:17:00,14:18:00,14:19:00,14:20:00,14:21:00,14:22:00,14:23:00,14:24:00,14:25:00,14:26:00,14:27:00,14:28:00,14:29:00,14:30:00,14:31:00,14:32:00,14:33:00,14:34:00,14:35:00,14:36:00,14:37:00,14:38:00,14:39:00,14:40:00,14:41:00,14:42:00,14:43:00,14:44:00,14:45:00,14:46:00,14:47:00,14:48:00,14:49:00,14:50:00,14:51:00,14:52:00,14:53:00,14:54:00,14:55:00,14:56:00,14:57:00,14:58:00,14:59:00,15:00:00
trading_date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2026-08-24,1.744258,1.289708,1.415416,1.829061,1.324606,1.277820,1.566195,1.853477,2.305179,1.016840,1.259648,1.509946,1.742393,1.406119,1.547350,1.601059,1.534522,1.171147,1.024449,1.008470,0.825666,0.752665,1.045208,0.734660,0.674831,1.186070,1.004819,0.765645,0.555731,0.698132,0.649486,0.853783,0.380392,0.386980,0.456986,0.790199,0.780180,1.194377,0.680796,0.672397,0.780130,0.776411,0.625908,0.649472,0.686232,0.618888,0.942753,0.794710,0.641392,0.552403,0.601283,0.835045,1.157539,0.684292,0.601674,0.37352,0.329793,0.702213,0.617085,0.499771,0.580485,0.624383,0.632096,0.372577,0.463445,0.534768,0.393332,0.350651,0.475347,0.343854,0.354776,0.520965,0.422041,0.602834,0.566896,0.821231,1.241213,0.692657,0.558502,0.440429,0.490753,0.317928,0.566959,0.974792,0.427920,0.403312,0.394241,0.261891,0.425583,0.362175,0.518765,0.631752,0.587181,0.500494,0.346361,0.420230,0.430879,0.510452,0.691327,0.597084,0.583424,0.494546,0.476261,0.504851,0.598138,0.598917,0.626256,0.792806,0.646102,0.79866,0.543440,0.296791,0.282416,0.425662,0.440816,0.388901,0.284708,0.586330,0.523501,0.388449,0.607826,0.514235,0.372191,0.521984,0.66398,0.505246,0.267741,0.511142,0.867104,0.567416,0.327521,0.457808,0.745031,0.286410,0.281814,0.423749,0.776063,0.763483,0.552304,0.266205,0.329278,0.560830,0.530127,0.514417,0.553770,0.386754,0.471502,0.595358,0.445988,0.370595,0.426469,0.569955,0.426237,0.383473,0.663847,0.426211,0.431925,0.639160,0.670312,0.667775,0.461462,0.334019,0.391202,0.566960,0.632698,0.341323,0.433260,0.539762,0.703104,0.649609,0.518733,0.616214,0.412825,0.363391,0

回望日历天数 lookback:  60
Index([09:31:00, 09:32:00, 09:33:00, 09:34:00, 09:35:00, 09:36:00, 09:37:00, 09:38:00, 09:39:00, 09:40:00,
       ...
       14:51:00, 14:52:00, 14:53:00, 14:54:00, 14:55:00, 14:56:00, 14:57:00, 14:58:00, 14:59:00, 15:00:00], dtype='object', name='minute_slot', length=240)


minute_slot,09:31:00,09:32:00,09:33:00,09:34:00,09:35:00,09:36:00,09:37:00,09:38:00,09:39:00,09:40:00,09:41:00,09:42:00,09:43:00,09:44:00,09:45:00,09:46:00,09:47:00,09:48:00,09:49:00,09:50:00,09:51:00,09:52:00,09:53:00,09:54:00,09:55:00,09:56:00,09:57:00,09:58:00,09:59:00,10:00:00,10:01:00,10:02:00,10:03:00,10:04:00,10:05:00,10:06:00,10:07:00,10:08:00,10:09:00,10:10:00,10:11:00,10:12:00,10:13:00,10:14:00,10:15:00,10:16:00,10:17:00,10:18:00,10:19:00,10:20:00,10:21:00,10:22:00,10:23:00,10:24:00,10:25:00,10:26:00,10:27:00,10:28:00,10:29:00,10:30:00,10:31:00,10:32:00,10:33:00,10:34:00,10:35:00,10:36:00,10:37:00,10:38:00,10:39:00,10:40:00,10:41:00,10:42:00,10:43:00,10:44:00,10:45:00,10:46:00,10:47:00,10:48:00,10:49:00,10:50:00,10:51:00,10:52:00,10:53:00,10:54:00,10:55:00,10:56:00,10:57:00,10:58:00,10:59:00,11:00:00,11:01:00,11:02:00,11:03:00,11:04:00,11:05:00,11:06:00,11:07:00,11:08:00,11:09:00,11:10:00,11:11:00,11:12:00,11:13:00,11:14:00,11:15:00,11:16:00,11:17:00,11:18:00,11:19:00,11:20:00,11:21:00,11:22:00,11:23:00,11:24:00,11:25:00,11:26:00,11:27:00,11:28:00,11:29:00,11:30:00,13:01:00,13:02:00,13:03:00,13:04:00,13:05:00,13:06:00,13:07:00,13:08:00,13:09:00,13:10:00,13:11:00,13:12:00,13:13:00,13:14:00,13:15:00,13:16:00,13:17:00,13:18:00,13:19:00,13:20:00,13:21:00,13:22:00,13:23:00,13:24:00,13:25:00,13:26:00,13:27:00,13:28:00,13:29:00,13:30:00,13:31:00,13:32:00,13:33:00,13:34:00,13:35:00,13:36:00,13:37:00,13:38:00,13:39:00,13:40:00,13:41:00,13:42:00,13:43:00,13:44:00,13:45:00,13:46:00,13:47:00,13:48:00,13:49:00,13:50:00,13:51:00,13:52:00,13:53:00,13:54:00,13:55:00,13:56:00,13:57:00,13:58:00,13:59:00,14:00:00,14:01:00,14:02:00,14:03:00,14:04:00,14:05:00,14:06:00,14:07:00,14:08:00,14:09:00,14:10:00,14:11:00,14:12:00,14:13:00,14:14:00,14:15:00,14:16:00,14:17:00,14:18:00,14:19:00,14:20:00,14:21:00,14:22:00,14:23:00,14:24:00,14:25:00,14:26:00,14:27:00,14:28:00,14:29:00,14:30:00,14:31:00,14:32:00,14:33:00,14:34:00,14:35:00,14:36:00,14:37:00,14:38:00,14:39:00,14:40:00,14:41:00,14:42:00,14:43:00,14:44:00,14:45:00,14:46:00,14:47:00,14:48:00,14:49:00,14:50:00,14:51:00,14:52:00,14:53:00,14:54:00,14:55:00,14:56:00,14:57:00,14:58:00,14:59:00,15:00:00
trading_date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2026-08-24,2.044299,1.858066,1.377397,1.800049,1.620350,0.974613,1.356323,2.004835,2.251627,1.287397,1.579068,1.426745,1.225515,1.383644,1.403817,1.601059,1.616864,1.449164,1.024449,1.025722,0.919599,0.901077,0.778224,0.734660,0.957857,0.97903,0.853884,0.769608,0.830590,0.806276,0.686692,0.985933,0.763576,0.489932,0.689298,0.768007,0.812912,1.033878,0.651760,0.718861,0.743511,0.728336,0.565981,0.793706,0.686232,0.655743,0.810755,0.759612,0.718911,0.584931,0.626609,0.748077,0.820696,0.618139,0.515369,0.405318,0.433828,0.727796,0.756816,0.455579,0.472595,0.390462,0.632096,0.742852,0.457251,0.471309,0.417857,0.500059,0.507125,0.351790,0.547907,0.613228,0.576403,0.738185,0.606334,0.798813,0.645298,0.580854,0.528668,0.422928,0.513392,0.392124,0.590869,0.599977,0.457775,0.438344,0.433966,0.529559,0.478311,0.386283,0.429883,0.552935,0.562494,0.524530,0.462671,0.395621,0.407487,0.505294,0.633100,0.602903,0.518618,0.449471,0.478470,0.315518,0.569490,0.598917,0.659433,0.790702,0.524876,0.382347,0.423635,0.367908,0.462130,0.368284,0.486285,0.484577,0.372871,0.413963,0.378643,0.343025,0.804369,0.764154,0.443969,0.521984,0.663980,0.505246,0.366655,0.475974,0.654741,0.466991,0.370144,0.411649,0.67033,0.352102,0.311475,0.335403,0.415943,0.533392,0.455801,0.363546,0.407585,0.407511,0.373739,0.416061,0.488266,0.436619,0.483939,0.558121,0.463799,0.369435,0.576042,0.685472,0.503944,0.393692,0.541912,0.643493,0.393785,0.534704,0.534881,0.612177,0.579703,0.468192,0.521249,0.522541,0.431110,0.303919,0.413256,0.495451,0.608461,0.557821,0.321244,0.459383,0.435649,0.379224,

回望日历天数 lookback:  120
Index([09:31:00, 09:32:00, 09:33:00, 09:34:00, 09:35:00, 09:36:00, 09:37:00, 09:38:00, 09:39:00, 09:40:00,
       ...
       14:51:00, 14:52:00, 14:53:00, 14:54:00, 14:55:00, 14:56:00, 14:57:00, 14:58:00, 14:59:00, 15:00:00], dtype='object', name='minute_slot', length=240)


minute_slot,09:31:00,09:32:00,09:33:00,09:34:00,09:35:00,09:36:00,09:37:00,09:38:00,09:39:00,09:40:00,09:41:00,09:42:00,09:43:00,09:44:00,09:45:00,09:46:00,09:47:00,09:48:00,09:49:00,09:50:00,09:51:00,09:52:00,09:53:00,09:54:00,09:55:00,09:56:00,09:57:00,09:58:00,09:59:00,10:00:00,10:01:00,10:02:00,10:03:00,10:04:00,10:05:00,10:06:00,10:07:00,10:08:00,10:09:00,10:10:00,10:11:00,10:12:00,10:13:00,10:14:00,10:15:00,10:16:00,10:17:00,10:18:00,10:19:00,10:20:00,10:21:00,10:22:00,10:23:00,10:24:00,10:25:00,10:26:00,10:27:00,10:28:00,10:29:00,10:30:00,10:31:00,10:32:00,10:33:00,10:34:00,10:35:00,10:36:00,10:37:00,10:38:00,10:39:00,10:40:00,10:41:00,10:42:00,10:43:00,10:44:00,10:45:00,10:46:00,10:47:00,10:48:00,10:49:00,10:50:00,10:51:00,10:52:00,10:53:00,10:54:00,10:55:00,10:56:00,10:57:00,10:58:00,10:59:00,11:00:00,11:01:00,11:02:00,11:03:00,11:04:00,11:05:00,11:06:00,11:07:00,11:08:00,11:09:00,11:10:00,11:11:00,11:12:00,11:13:00,11:14:00,11:15:00,11:16:00,11:17:00,11:18:00,11:19:00,11:20:00,11:21:00,11:22:00,11:23:00,11:24:00,11:25:00,11:26:00,11:27:00,11:28:00,11:29:00,11:30:00,13:01:00,13:02:00,13:03:00,13:04:00,13:05:00,13:06:00,13:07:00,13:08:00,13:09:00,13:10:00,13:11:00,13:12:00,13:13:00,13:14:00,13:15:00,13:16:00,13:17:00,13:18:00,13:19:00,13:20:00,13:21:00,13:22:00,13:23:00,13:24:00,13:25:00,13:26:00,13:27:00,13:28:00,13:29:00,13:30:00,13:31:00,13:32:00,13:33:00,13:34:00,13:35:00,13:36:00,13:37:00,13:38:00,13:39:00,13:40:00,13:41:00,13:42:00,13:43:00,13:44:00,13:45:00,13:46:00,13:47:00,13:48:00,13:49:00,13:50:00,13:51:00,13:52:00,13:53:00,13:54:00,13:55:00,13:56:00,13:57:00,13:58:00,13:59:00,14:00:00,14:01:00,14:02:00,14:03:00,14:04:00,14:05:00,14:06:00,14:07:00,14:08:00,14:09:00,14:10:00,14:11:00,14:12:00,14:13:00,14:14:00,14:15:00,14:16:00,14:17:00,14:18:00,14:19:00,14:20:00,14:21:00,14:22:00,14:23:00,14:24:00,14:25:00,14:26:00,14:27:00,14:28:00,14:29:00,14:30:00,14:31:00,14:32:00,14:33:00,14:34:00,14:35:00,14:36:00,14:37:00,14:38:00,14:39:00,14:40:00,14:41:00,14:42:00,14:43:00,14:44:00,14:45:00,14:46:00,14:47:00,14:48:00,14:49:00,14:50:00,14:51:00,14:52:00,14:53:00,14:54:00,14:55:00,14:56:00,14:57:00,14:58:00,14:59:00,15:00:00
trading_date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2026-08-24,2.461031,2.548271,1.596765,1.993283,1.939121,1.416912,1.507239,2.295070,2.210346,1.442481,1.828928,1.491879,1.450033,1.407575,1.233703,1.479561,1.453214,1.184075,1.064235,1.159671,1.035516,1.168313,1.023168,0.854232,0.853720,0.903255,0.802911,0.778061,0.875017,0.919379,0.796584,0.941943,0.763377,0.789857,0.818759,0.790199,0.812912,0.988711,0.813280,0.748996,0.784080,0.728336,0.565981,0.784418,0.602024,0.635376,0.818239,0.719818,0.706464,0.541236,0.525645,0.695966,0.740905,0.518817,0.510379,0.452186,0.422754,0.550521,0.611642,0.424370,0.462953,0.394534,0.641795,0.691250,0.518411,0.584213,0.450950,0.476662,0.555290,0.586565,0.487545,0.568940,0.536982,0.57383,0.519863,0.686219,0.638566,0.587053,0.475692,0.454649,0.440659,0.317928,0.531958,0.484053,0.460370,0.572429,0.487307,0.486630,0.392847,0.367032,0.479585,0.719646,0.596039,0.543709,0.422941,0.390533,0.456301,0.501955,0.521964,0.512163,0.383761,0.384088,0.439787,0.312455,0.490309,0.495657,0.580575,0.611017,0.475404,0.373485,0.451762,0.458963,0.565192,0.368284,0.528697,0.449591,0.364562,0.422584,0.389914,0.340894,0.984588,1.076222,0.674364,0.469382,0.564136,0.607996,0.538484,0.475974,0.432149,0.521896,0.439941,0.457642,0.576299,0.301021,0.281978,0.324359,0.330988,0.481545,0.394957,0.314041,0.266142,0.389700,0.391063,0.416061,0.385627,0.357607,0.406982,0.408348,0.383736,0.310532,0.440824,0.590289,0.49014,0.357167,0.505215,0.578978,0.456628,0.474328,0.375590,0.426718,0.493084,0.427586,0.435134,0.316922,0.404814,0.401483,0.413256,0.378137,0.421155,0.347230,0.340442,0.419733,0.412825,0.391674,

回望日历天数 lookback:  360
Index([09:31:00, 09:32:00, 09:33:00, 09:34:00, 09:35:00, 09:36:00, 09:37:00, 09:38:00, 09:39:00, 09:40:00,
       ...
       14:51:00, 14:52:00, 14:53:00, 14:54:00, 14:55:00, 14:56:00, 14:57:00, 14:58:00, 14:59:00, 15:00:00], dtype='object', name='minute_slot', length=240)


minute_slot,09:31:00,09:32:00,09:33:00,09:34:00,09:35:00,09:36:00,09:37:00,09:38:00,09:39:00,09:40:00,09:41:00,09:42:00,09:43:00,09:44:00,09:45:00,09:46:00,09:47:00,09:48:00,09:49:00,09:50:00,09:51:00,09:52:00,09:53:00,09:54:00,09:55:00,09:56:00,09:57:00,09:58:00,09:59:00,10:00:00,10:01:00,10:02:00,10:03:00,10:04:00,10:05:00,10:06:00,10:07:00,10:08:00,10:09:00,10:10:00,10:11:00,10:12:00,10:13:00,10:14:00,10:15:00,10:16:00,10:17:00,10:18:00,10:19:00,10:20:00,10:21:00,10:22:00,10:23:00,10:24:00,10:25:00,10:26:00,10:27:00,10:28:00,10:29:00,10:30:00,10:31:00,10:32:00,10:33:00,10:34:00,10:35:00,10:36:00,10:37:00,10:38:00,10:39:00,10:40:00,10:41:00,10:42:00,10:43:00,10:44:00,10:45:00,10:46:00,10:47:00,10:48:00,10:49:00,10:50:00,10:51:00,10:52:00,10:53:00,10:54:00,10:55:00,10:56:00,10:57:00,10:58:00,10:59:00,11:00:00,11:01:00,11:02:00,11:03:00,11:04:00,11:05:00,11:06:00,11:07:00,11:08:00,11:09:00,11:10:00,11:11:00,11:12:00,11:13:00,11:14:00,11:15:00,11:16:00,11:17:00,11:18:00,11:19:00,11:20:00,11:21:00,11:22:00,11:23:00,11:24:00,11:25:00,11:26:00,11:27:00,11:28:00,11:29:00,11:30:00,13:01:00,13:02:00,13:03:00,13:04:00,13:05:00,13:06:00,13:07:00,13:08:00,13:09:00,13:10:00,13:11:00,13:12:00,13:13:00,13:14:00,13:15:00,13:16:00,13:17:00,13:18:00,13:19:00,13:20:00,13:21:00,13:22:00,13:23:00,13:24:00,13:25:00,13:26:00,13:27:00,13:28:00,13:29:00,13:30:00,13:31:00,13:32:00,13:33:00,13:34:00,13:35:00,13:36:00,13:37:00,13:38:00,13:39:00,13:40:00,13:41:00,13:42:00,13:43:00,13:44:00,13:45:00,13:46:00,13:47:00,13:48:00,13:49:00,13:50:00,13:51:00,13:52:00,13:53:00,13:54:00,13:55:00,13:56:00,13:57:00,13:58:00,13:59:00,14:00:00,14:01:00,14:02:00,14:03:00,14:04:00,14:05:00,14:06:00,14:07:00,14:08:00,14:09:00,14:10:00,14:11:00,14:12:00,14:13:00,14:14:00,14:15:00,14:16:00,14:17:00,14:18:00,14:19:00,14:20:00,14:21:00,14:22:00,14:23:00,14:24:00,14:25:00,14:26:00,14:27:00,14:28:00,14:29:00,14:30:00,14:31:00,14:32:00,14:33:00,14:34:00,14:35:00,14:36:00,14:37:00,14:38:00,14:39:00,14:40:00,14:41:00,14:42:00,14:43:00,14:44:00,14:45:00,14:46:00,14:47:00,14:48:00,14:49:00,14:50:00,14:51:00,14:52:00,14:53:00,14:54:00,14:55:00,14:56:00,14:57:00,14:58:00,14:59:00,15:00:00
trading_date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2026-08-24,4.640836,4.043054,2.233957,1.996005,2.007368,1.770184,1.798330,1.872153,1.605569,1.308700,1.539837,1.546812,1.325164,1.181679,1.138830,1.134066,0.95494,0.902009,0.949703,1.007458,1.159837,0.985600,0.933431,0.788454,0.856980,0.764989,0.652661,0.737873,0.857997,0.856359,0.903944,0.825217,0.711377,0.759228,0.741035,0.749594,0.692049,0.738717,0.723870,0.702001,0.594905,0.613976,0.556134,0.599432,0.569158,0.631492,0.655935,0.635243,0.664991,0.549113,0.543165,0.620537,0.571243,0.480227,0.530539,0.446859,0.447914,0.472076,0.436268,0.462144,0.490447,0.476480,0.636076,0.575030,0.508944,0.492361,0.448330,0.437489,0.423750,0.410713,0.386306,0.404507,0.462581,0.469995,0.407563,0.440144,0.406011,0.409533,0.361438,0.335304,0.387276,0.345838,0.418142,0.494083,0.407982,0.411255,0.435110,0.426281,0.355377,0.396579,0.463455,0.541603,0.503848,0.496555,0.446352,0.373886,0.363954,0.423392,0.428034,0.424993,0.375753,0.400694,0.437980,0.358473,0.429783,0.448429,0.560209,0.495558,0.465537,0.422381,0.465372,0.495649,0.405916,0.457257,0.481364,0.448141,0.391926,0.412320,0.357727,0.37863,0.809133,0.838893,0.569660,0.523669,0.500498,0.483428,0.425366,0.411975,0.395754,0.394260,0.357782,0.355721,0.390028,0.295157,0.311082,0.322372,0.356995,0.367114,0.360590,0.346148,0.264808,0.330852,0.331702,0.317189,0.325083,0.361757,0.298598,0.278354,0.328609,0.262573,0.316416,0.436442,0.414582,0.357370,0.444825,0.410552,0.359364,0.307652,0.302770,0.310145,0.344516,0.335208,0.348490,0.289259,0.350024,0.390153,0.365944,0.298393,0.362966,0.354210,0.316865,0.344223,0.336685,0.351295,

### 季节曲线归一化：估计混合贡献因子

对此前 $L$ 个交易日的 $x^{\mathrm{mix}}$，逐分钟取中位数；
再将得到的历史曲线归一到日内均值 1：

$$
\widehat g^{\mathrm{mix},(L)}_{t,i}
=
\frac{
\operatorname{median}_{h=t-L,\ldots,t-1}x^{\mathrm{mix}}_{h,i}
}{
M^{-1}\sum_{j=1}^{M}
\operatorname{median}_{h=t-L,\ldots,t-1}x^{\mathrm{mix}}_{h,j}
}.
$$

帽号表示它由历史样本估计得到，$(L)$ 表示所用回望窗口。
逐分钟取中位数后，整条曲线的均值未必仍为 1，因此需要第二次归一化。
大于 1 表示该分钟的历史典型相对贡献高于这条曲线的平均水平，
小于 1 表示低于平均水平。归一化只使用历史估计曲线，不使用目标日的实际贡献。

$\widehat g^{\mathrm{mix},(L)}$ 是**混合贡献季节因子**。
其估计对象与收益的方差周期 $f^2$ 有关，但两者不能由此定义直接画等号。

#### Cell 9：把历史中位数曲线归一化为日内分钟均值 1。

历史截至前一交易日由上面的 `shift(1)` 保证；当前结果未单独保存历史截止日期字段。

In [38]:
seasonality_by_lookback = {}

for lookback in lookback_windows:
    rolling_median_matrix = rolling_median_by_lookback[lookback]
    curve_mean_series = rolling_median_matrix.mean(axis=1)

    seasonality_by_lookback[lookback] = rolling_median_matrix.div(
        curve_mean_series.where(curve_mean_series > 0),
        axis=0,
    )

In [41]:
print(seasonality_by_lookback.keys())
for key, value in  seasonality_by_lookback.items():
    print("回望日历天数 lookback: ", key)
    print(value.columns)
    display(value.tail())

dict_keys([30, 60, 120, 360])
回望日历天数 lookback:  30
Index([09:31:00, 09:32:00, 09:33:00, 09:34:00, 09:35:00, 09:36:00, 09:37:00, 09:38:00, 09:39:00, 09:40:00,
       ...
       14:51:00, 14:52:00, 14:53:00, 14:54:00, 14:55:00, 14:56:00, 14:57:00, 14:58:00, 14:59:00, 15:00:00], dtype='object', name='minute_slot', length=240)


minute_slot,09:31:00,09:32:00,09:33:00,09:34:00,09:35:00,09:36:00,09:37:00,09:38:00,09:39:00,09:40:00,09:41:00,09:42:00,09:43:00,09:44:00,09:45:00,09:46:00,09:47:00,09:48:00,09:49:00,09:50:00,09:51:00,09:52:00,09:53:00,09:54:00,09:55:00,09:56:00,09:57:00,09:58:00,09:59:00,10:00:00,10:01:00,10:02:00,10:03:00,10:04:00,10:05:00,10:06:00,10:07:00,10:08:00,10:09:00,10:10:00,10:11:00,10:12:00,10:13:00,10:14:00,10:15:00,10:16:00,10:17:00,10:18:00,10:19:00,10:20:00,10:21:00,10:22:00,10:23:00,10:24:00,10:25:00,10:26:00,10:27:00,10:28:00,10:29:00,10:30:00,10:31:00,10:32:00,10:33:00,10:34:00,10:35:00,10:36:00,10:37:00,10:38:00,10:39:00,10:40:00,10:41:00,10:42:00,10:43:00,10:44:00,10:45:00,10:46:00,10:47:00,10:48:00,10:49:00,10:50:00,10:51:00,10:52:00,10:53:00,10:54:00,10:55:00,10:56:00,10:57:00,10:58:00,10:59:00,11:00:00,11:01:00,11:02:00,11:03:00,11:04:00,11:05:00,11:06:00,11:07:00,11:08:00,11:09:00,11:10:00,11:11:00,11:12:00,11:13:00,11:14:00,11:15:00,11:16:00,11:17:00,11:18:00,11:19:00,11:20:00,11:21:00,11:22:00,11:23:00,11:24:00,11:25:00,11:26:00,11:27:00,11:28:00,11:29:00,11:30:00,13:01:00,13:02:00,13:03:00,13:04:00,13:05:00,13:06:00,13:07:00,13:08:00,13:09:00,13:10:00,13:11:00,13:12:00,13:13:00,13:14:00,13:15:00,13:16:00,13:17:00,13:18:00,13:19:00,13:20:00,13:21:00,13:22:00,13:23:00,13:24:00,13:25:00,13:26:00,13:27:00,13:28:00,13:29:00,13:30:00,13:31:00,13:32:00,13:33:00,13:34:00,13:35:00,13:36:00,13:37:00,13:38:00,13:39:00,13:40:00,13:41:00,13:42:00,13:43:00,13:44:00,13:45:00,13:46:00,13:47:00,13:48:00,13:49:00,13:50:00,13:51:00,13:52:00,13:53:00,13:54:00,13:55:00,13:56:00,13:57:00,13:58:00,13:59:00,14:00:00,14:01:00,14:02:00,14:03:00,14:04:00,14:05:00,14:06:00,14:07:00,14:08:00,14:09:00,14:10:00,14:11:00,14:12:00,14:13:00,14:14:00,14:15:00,14:16:00,14:17:00,14:18:00,14:19:00,14:20:00,14:21:00,14:22:00,14:23:00,14:24:00,14:25:00,14:26:00,14:27:00,14:28:00,14:29:00,14:30:00,14:31:00,14:32:00,14:33:00,14:34:00,14:35:00,14:36:00,14:37:00,14:38:00,14:39:00,14:40:00,14:41:00,14:42:00,14:43:00,14:44:00,14:45:00,14:46:00,14:47:00,14:48:00,14:49:00,14:50:00,14:51:00,14:52:00,14:53:00,14:54:00,14:55:00,14:56:00,14:57:00,14:58:00,14:59:00,15:00:00
trading_date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2026-08-24,2.949197,2.180643,2.393191,3.092583,2.239649,2.160542,2.648128,3.133866,3.897606,1.719277,2.129818,2.553023,2.946045,2.377471,2.616266,2.707076,2.594575,1.980180,1.732143,1.705125,1.396039,1.272609,1.767241,1.242166,1.141006,2.005411,1.698951,1.294555,0.939632,1.180404,1.098154,1.443580,0.643168,0.654308,0.772674,1.336071,1.319132,2.019457,1.151092,1.136892,1.319046,1.312758,1.058288,1.098129,1.160283,1.046418,1.594011,1.343699,1.084469,0.934006,1.016651,1.411898,1.957172,1.157003,1.017313,0.631549,0.557615,1.187304,1.043369,0.845014,0.981487,1.055709,1.068750,0.629954,0.783595,0.904187,0.665047,0.592882,0.803718,0.581390,0.599857,0.880850,0.713588,1.019274,0.958511,1.388541,2.098647,1.171147,0.944317,0.744679,0.829766,0.537553,0.958616,1.648183,0.723529,0.681922,0.666584,0.442807,0.719578,0.612367,0.877131,1.068169,0.992807,0.846237,0.585628,0.710527,0.728532,0.863074,1.168899,1.009552,0.986456,0.836181,0.805263,0.853604,1.011334,1.012651,1.058876,1.340479,1.092431,1.350377,0.918850,0.501816,0.477510,0.719712,0.745333,0.657556,0.481385,0.991369,0.885138,0.656790,1.027715,0.869471,0.629303,0.882573,1.122660,0.854271,0.452697,0.864241,1.466102,0.959389,0.553774,0.774063,1.259702,0.484263,0.476492,0.716477,1.312170,1.290899,0.933838,0.450101,0.556744,0.948254,0.896340,0.869779,0.936316,0.653926,0.797218,1.006634,0.754078,0.626603,0.721076,0.963682,0.720684,0.648378,1.122436,0.720640,0.730300,1.080694,1.133366,1.129076,0.780242,0.564761,0.661446,0.958619,1.069768,0.577110,0.732558,0.912631,1.188811,1.098361,0.877075,1.041897,0.698005,0.61442

回望日历天数 lookback:  60
Index([09:31:00, 09:32:00, 09:33:00, 09:34:00, 09:35:00, 09:36:00, 09:37:00, 09:38:00, 09:39:00, 09:40:00,
       ...
       14:51:00, 14:52:00, 14:53:00, 14:54:00, 14:55:00, 14:56:00, 14:57:00, 14:58:00, 14:59:00, 15:00:00], dtype='object', name='minute_slot', length=240)


minute_slot,09:31:00,09:32:00,09:33:00,09:34:00,09:35:00,09:36:00,09:37:00,09:38:00,09:39:00,09:40:00,09:41:00,09:42:00,09:43:00,09:44:00,09:45:00,09:46:00,09:47:00,09:48:00,09:49:00,09:50:00,09:51:00,09:52:00,09:53:00,09:54:00,09:55:00,09:56:00,09:57:00,09:58:00,09:59:00,10:00:00,10:01:00,10:02:00,10:03:00,10:04:00,10:05:00,10:06:00,10:07:00,10:08:00,10:09:00,10:10:00,10:11:00,10:12:00,10:13:00,10:14:00,10:15:00,10:16:00,10:17:00,10:18:00,10:19:00,10:20:00,10:21:00,10:22:00,10:23:00,10:24:00,10:25:00,10:26:00,10:27:00,10:28:00,10:29:00,10:30:00,10:31:00,10:32:00,10:33:00,10:34:00,10:35:00,10:36:00,10:37:00,10:38:00,10:39:00,10:40:00,10:41:00,10:42:00,10:43:00,10:44:00,10:45:00,10:46:00,10:47:00,10:48:00,10:49:00,10:50:00,10:51:00,10:52:00,10:53:00,10:54:00,10:55:00,10:56:00,10:57:00,10:58:00,10:59:00,11:00:00,11:01:00,11:02:00,11:03:00,11:04:00,11:05:00,11:06:00,11:07:00,11:08:00,11:09:00,11:10:00,11:11:00,11:12:00,11:13:00,11:14:00,11:15:00,11:16:00,11:17:00,11:18:00,11:19:00,11:20:00,11:21:00,11:22:00,11:23:00,11:24:00,11:25:00,11:26:00,11:27:00,11:28:00,11:29:00,11:30:00,13:01:00,13:02:00,13:03:00,13:04:00,13:05:00,13:06:00,13:07:00,13:08:00,13:09:00,13:10:00,13:11:00,13:12:00,13:13:00,13:14:00,13:15:00,13:16:00,13:17:00,13:18:00,13:19:00,13:20:00,13:21:00,13:22:00,13:23:00,13:24:00,13:25:00,13:26:00,13:27:00,13:28:00,13:29:00,13:30:00,13:31:00,13:32:00,13:33:00,13:34:00,13:35:00,13:36:00,13:37:00,13:38:00,13:39:00,13:40:00,13:41:00,13:42:00,13:43:00,13:44:00,13:45:00,13:46:00,13:47:00,13:48:00,13:49:00,13:50:00,13:51:00,13:52:00,13:53:00,13:54:00,13:55:00,13:56:00,13:57:00,13:58:00,13:59:00,14:00:00,14:01:00,14:02:00,14:03:00,14:04:00,14:05:00,14:06:00,14:07:00,14:08:00,14:09:00,14:10:00,14:11:00,14:12:00,14:13:00,14:14:00,14:15:00,14:16:00,14:17:00,14:18:00,14:19:00,14:20:00,14:21:00,14:22:00,14:23:00,14:24:00,14:25:00,14:26:00,14:27:00,14:28:00,14:29:00,14:30:00,14:31:00,14:32:00,14:33:00,14:34:00,14:35:00,14:36:00,14:37:00,14:38:00,14:39:00,14:40:00,14:41:00,14:42:00,14:43:00,14:44:00,14:45:00,14:46:00,14:47:00,14:48:00,14:49:00,14:50:00,14:51:00,14:52:00,14:53:00,14:54:00,14:55:00,14:56:00,14:57:00,14:58:00,14:59:00,15:00:00
trading_date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2026-08-24,3.501789,3.182782,2.359418,3.083400,2.775584,1.669467,2.323319,3.434190,3.856933,2.205251,2.704870,2.443947,2.099250,2.370118,2.404674,2.742540,2.769613,2.482351,1.754834,1.757014,1.575231,1.543503,1.333062,1.258439,1.640766,1.677034,1.462664,1.318303,1.422762,1.381114,1.176272,1.688858,1.307970,0.839231,1.180735,1.315560,1.392480,1.770985,1.116434,1.231376,1.273600,1.247606,0.969500,1.359581,1.175483,1.123257,1.388786,1.301180,1.231461,1.001959,1.073352,1.281422,1.405815,1.058843,0.882804,0.694290,0.743127,1.246681,1.296390,0.780385,0.809534,0.668844,1.082751,1.272471,0.783251,0.807330,0.715771,0.856578,0.868682,0.602600,0.938540,1.050431,0.987351,1.264476,1.038621,1.368330,1.105365,0.994977,0.905583,0.724456,0.879416,0.671690,1.012132,1.027733,0.784147,0.750863,0.743363,0.907109,0.819324,0.661685,0.736370,0.947153,0.963526,0.898496,0.792534,0.677680,0.698007,0.865545,1.084471,1.032745,0.888369,0.769923,0.819597,0.540468,0.975510,1.025917,1.129579,1.354435,0.899089,0.654943,0.725667,0.630210,0.791607,0.630854,0.832984,0.830058,0.638711,0.709099,0.648597,0.587586,1.377847,1.308961,0.760499,0.894135,1.137367,0.865462,0.628063,0.815321,1.121540,0.799933,0.634039,0.705135,1.148244,0.603135,0.533542,0.574529,0.712492,0.913676,0.780765,0.622738,0.698175,0.698048,0.640197,0.712693,0.836378,0.747908,0.828965,0.956035,0.794467,0.632825,0.986733,1.174182,0.863233,0.674377,0.928270,1.102273,0.674535,0.915922,0.916226,1.048631,0.993004,0.801992,0.892876,0.895088,0.738472,0.520599,0.707889,0.848685,1.042266,0.955522,0.550277,0.786901,0.746247,0.64959

回望日历天数 lookback:  120
Index([09:31:00, 09:32:00, 09:33:00, 09:34:00, 09:35:00, 09:36:00, 09:37:00, 09:38:00, 09:39:00, 09:40:00,
       ...
       14:51:00, 14:52:00, 14:53:00, 14:54:00, 14:55:00, 14:56:00, 14:57:00, 14:58:00, 14:59:00, 15:00:00], dtype='object', name='minute_slot', length=240)


minute_slot,09:31:00,09:32:00,09:33:00,09:34:00,09:35:00,09:36:00,09:37:00,09:38:00,09:39:00,09:40:00,09:41:00,09:42:00,09:43:00,09:44:00,09:45:00,09:46:00,09:47:00,09:48:00,09:49:00,09:50:00,09:51:00,09:52:00,09:53:00,09:54:00,09:55:00,09:56:00,09:57:00,09:58:00,09:59:00,10:00:00,10:01:00,10:02:00,10:03:00,10:04:00,10:05:00,10:06:00,10:07:00,10:08:00,10:09:00,10:10:00,10:11:00,10:12:00,10:13:00,10:14:00,10:15:00,10:16:00,10:17:00,10:18:00,10:19:00,10:20:00,10:21:00,10:22:00,10:23:00,10:24:00,10:25:00,10:26:00,10:27:00,10:28:00,10:29:00,10:30:00,10:31:00,10:32:00,10:33:00,10:34:00,10:35:00,10:36:00,10:37:00,10:38:00,10:39:00,10:40:00,10:41:00,10:42:00,10:43:00,10:44:00,10:45:00,10:46:00,10:47:00,10:48:00,10:49:00,10:50:00,10:51:00,10:52:00,10:53:00,10:54:00,10:55:00,10:56:00,10:57:00,10:58:00,10:59:00,11:00:00,11:01:00,11:02:00,11:03:00,11:04:00,11:05:00,11:06:00,11:07:00,11:08:00,11:09:00,11:10:00,11:11:00,11:12:00,11:13:00,11:14:00,11:15:00,11:16:00,11:17:00,11:18:00,11:19:00,11:20:00,11:21:00,11:22:00,11:23:00,11:24:00,11:25:00,11:26:00,11:27:00,11:28:00,11:29:00,11:30:00,13:01:00,13:02:00,13:03:00,13:04:00,13:05:00,13:06:00,13:07:00,13:08:00,13:09:00,13:10:00,13:11:00,13:12:00,13:13:00,13:14:00,13:15:00,13:16:00,13:17:00,13:18:00,13:19:00,13:20:00,13:21:00,13:22:00,13:23:00,13:24:00,13:25:00,13:26:00,13:27:00,13:28:00,13:29:00,13:30:00,13:31:00,13:32:00,13:33:00,13:34:00,13:35:00,13:36:00,13:37:00,13:38:00,13:39:00,13:40:00,13:41:00,13:42:00,13:43:00,13:44:00,13:45:00,13:46:00,13:47:00,13:48:00,13:49:00,13:50:00,13:51:00,13:52:00,13:53:00,13:54:00,13:55:00,13:56:00,13:57:00,13:58:00,13:59:00,14:00:00,14:01:00,14:02:00,14:03:00,14:04:00,14:05:00,14:06:00,14:07:00,14:08:00,14:09:00,14:10:00,14:11:00,14:12:00,14:13:00,14:14:00,14:15:00,14:16:00,14:17:00,14:18:00,14:19:00,14:20:00,14:21:00,14:22:00,14:23:00,14:24:00,14:25:00,14:26:00,14:27:00,14:28:00,14:29:00,14:30:00,14:31:00,14:32:00,14:33:00,14:34:00,14:35:00,14:36:00,14:37:00,14:38:00,14:39:00,14:40:00,14:41:00,14:42:00,14:43:00,14:44:00,14:45:00,14:46:00,14:47:00,14:48:00,14:49:00,14:50:00,14:51:00,14:52:00,14:53:00,14:54:00,14:55:00,14:56:00,14:57:00,14:58:00,14:59:00,15:00:00
trading_date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2026-08-24,4.285243,4.437149,2.780349,3.470782,3.376472,2.467182,2.624464,3.996264,3.848740,2.511704,3.184600,2.597717,2.524854,2.450924,2.148172,2.576269,2.530393,2.061757,1.853087,2.019264,1.803081,2.034312,1.781580,1.487422,1.486531,1.572783,1.398059,1.354790,1.523614,1.600859,1.387043,1.640148,1.329221,1.375329,1.425654,1.375924,1.415473,1.721582,1.416114,1.304181,1.365271,1.268207,0.985509,1.365859,1.048267,1.106342,1.424750,1.253375,1.230122,0.942422,0.915273,1.211843,1.290092,0.903384,0.888691,0.787364,0.736116,0.958588,1.065015,0.738930,0.806111,0.686977,1.117518,1.203632,0.902678,1.017253,0.785212,0.829982,0.966893,1.021350,0.848933,0.990661,0.935014,0.999176,0.905206,1.194870,1.111895,1.022199,0.828294,0.791653,0.767292,0.553588,0.926266,0.842853,0.801614,0.996735,0.848518,0.847339,0.684041,0.639091,0.835073,1.253076,1.037847,0.946728,0.736441,0.680011,0.794529,0.874024,0.908864,0.891799,0.668220,0.668790,0.765774,0.544059,0.853744,0.863057,1.010920,1.063926,0.827791,0.650327,0.786626,0.799164,0.984134,0.641271,0.920588,0.782845,0.634789,0.735820,0.678934,0.593578,1.714403,1.873960,1.174229,0.817306,0.982296,1.058666,0.937630,0.828784,0.752474,0.908745,0.766042,0.796864,1.003475,0.524149,0.490990,0.564786,0.576329,0.838485,0.687715,0.546820,0.463416,0.678561,0.680934,0.724461,0.671469,0.622679,0.708654,0.711031,0.668177,0.540710,0.767580,1.027834,0.853451,0.621913,0.879699,1.008139,0.795098,0.825918,0.653992,0.743017,0.858576,0.744530,0.757673,0.551837,0.704877,0.699077,0.719578,0.658426,0.733332,0.604611,0.592790,0.730855,0.718826,0.68199

回望日历天数 lookback:  360
Index([09:31:00, 09:32:00, 09:33:00, 09:34:00, 09:35:00, 09:36:00, 09:37:00, 09:38:00, 09:39:00, 09:40:00,
       ...
       14:51:00, 14:52:00, 14:53:00, 14:54:00, 14:55:00, 14:56:00, 14:57:00, 14:58:00, 14:59:00, 15:00:00], dtype='object', name='minute_slot', length=240)


minute_slot,09:31:00,09:32:00,09:33:00,09:34:00,09:35:00,09:36:00,09:37:00,09:38:00,09:39:00,09:40:00,09:41:00,09:42:00,09:43:00,09:44:00,09:45:00,09:46:00,09:47:00,09:48:00,09:49:00,09:50:00,09:51:00,09:52:00,09:53:00,09:54:00,09:55:00,09:56:00,09:57:00,09:58:00,09:59:00,10:00:00,10:01:00,10:02:00,10:03:00,10:04:00,10:05:00,10:06:00,10:07:00,10:08:00,10:09:00,10:10:00,10:11:00,10:12:00,10:13:00,10:14:00,10:15:00,10:16:00,10:17:00,10:18:00,10:19:00,10:20:00,10:21:00,10:22:00,10:23:00,10:24:00,10:25:00,10:26:00,10:27:00,10:28:00,10:29:00,10:30:00,10:31:00,10:32:00,10:33:00,10:34:00,10:35:00,10:36:00,10:37:00,10:38:00,10:39:00,10:40:00,10:41:00,10:42:00,10:43:00,10:44:00,10:45:00,10:46:00,10:47:00,10:48:00,10:49:00,10:50:00,10:51:00,10:52:00,10:53:00,10:54:00,10:55:00,10:56:00,10:57:00,10:58:00,10:59:00,11:00:00,11:01:00,11:02:00,11:03:00,11:04:00,11:05:00,11:06:00,11:07:00,11:08:00,11:09:00,11:10:00,11:11:00,11:12:00,11:13:00,11:14:00,11:15:00,11:16:00,11:17:00,11:18:00,11:19:00,11:20:00,11:21:00,11:22:00,11:23:00,11:24:00,11:25:00,11:26:00,11:27:00,11:28:00,11:29:00,11:30:00,13:01:00,13:02:00,13:03:00,13:04:00,13:05:00,13:06:00,13:07:00,13:08:00,13:09:00,13:10:00,13:11:00,13:12:00,13:13:00,13:14:00,13:15:00,13:16:00,13:17:00,13:18:00,13:19:00,13:20:00,13:21:00,13:22:00,13:23:00,13:24:00,13:25:00,13:26:00,13:27:00,13:28:00,13:29:00,13:30:00,13:31:00,13:32:00,13:33:00,13:34:00,13:35:00,13:36:00,13:37:00,13:38:00,13:39:00,13:40:00,13:41:00,13:42:00,13:43:00,13:44:00,13:45:00,13:46:00,13:47:00,13:48:00,13:49:00,13:50:00,13:51:00,13:52:00,13:53:00,13:54:00,13:55:00,13:56:00,13:57:00,13:58:00,13:59:00,14:00:00,14:01:00,14:02:00,14:03:00,14:04:00,14:05:00,14:06:00,14:07:00,14:08:00,14:09:00,14:10:00,14:11:00,14:12:00,14:13:00,14:14:00,14:15:00,14:16:00,14:17:00,14:18:00,14:19:00,14:20:00,14:21:00,14:22:00,14:23:00,14:24:00,14:25:00,14:26:00,14:27:00,14:28:00,14:29:00,14:30:00,14:31:00,14:32:00,14:33:00,14:34:00,14:35:00,14:36:00,14:37:00,14:38:00,14:39:00,14:40:00,14:41:00,14:42:00,14:43:00,14:44:00,14:45:00,14:46:00,14:47:00,14:48:00,14:49:00,14:50:00,14:51:00,14:52:00,14:53:00,14:54:00,14:55:00,14:56:00,14:57:00,14:58:00,14:59:00,15:00:00
trading_date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2026-08-24,8.537527,7.437814,4.109705,3.671957,3.692861,3.256523,3.308302,3.444110,2.953689,2.407553,2.832766,2.845597,2.437842,2.173879,2.095051,2.086287,1.756758,1.659384,1.747124,1.853372,2.133697,1.813162,1.717189,1.450481,1.576545,1.407314,1.200670,1.357429,1.578417,1.575403,1.662944,1.518112,1.308686,1.396716,1.363247,1.378992,1.273130,1.358983,1.331669,1.291438,1.094419,1.129503,1.023093,1.102746,1.047053,1.161727,1.206693,1.168626,1.223352,1.010177,0.999235,1.141572,1.050888,0.883450,0.976008,0.822066,0.824006,0.868457,0.802581,0.850185,0.902253,0.876557,1.170159,1.057856,0.936281,0.905772,0.824772,0.804827,0.779553,0.755569,0.710668,0.744152,0.850988,0.864628,0.749775,0.809711,0.746919,0.753399,0.664920,0.616843,0.712454,0.636222,0.769235,0.908942,0.750545,0.756565,0.800451,0.784208,0.653770,0.729568,0.852596,0.996360,0.926905,0.913488,0.821133,0.687821,0.669548,0.778893,0.787435,0.781839,0.691254,0.737138,0.805732,0.659466,0.790652,0.824953,1.030590,0.911654,0.856427,0.777034,0.856124,0.911822,0.746745,0.841193,0.885543,0.824423,0.721008,0.758525,0.658094,0.696549,1.488523,1.543273,1.047976,0.963368,0.920742,0.889339,0.782526,0.757890,0.728050,0.725302,0.658195,0.654402,0.717516,0.542986,0.572284,0.593053,0.656747,0.675362,0.663360,0.636792,0.487155,0.608653,0.610216,0.583518,0.598039,0.665507,0.549317,0.512074,0.604526,0.483043,0.582095,0.802902,0.762686,0.657438,0.818324,0.755274,0.661104,0.565972,0.556992,0.570560,0.633790,0.616666,0.641100,0.532136,0.643922,0.717745,0.673209,0.548939,0.667732,0.651623,0.582922,0.633250,0.619383,0.64626

### 使用混合贡献季节因子

当前代码保留两种直接调整构造：

$$
c^{\mathrm{mix,adj},(L)}_{t,i}
=
\frac{c^{\mathrm{mix}}_{t,i}}
     {\widehat g^{\mathrm{mix},(L)}_{t,i}},
\qquad
r^{\mathrm{adj,mix},(L)}_{t,i}
=
\frac{r_{t,i}}
     {\sqrt{\widehat g^{\mathrm{mix},(L)}_{t,i}}}.
$$

它们分别对应 `variance_adjusted` 和 `return_adjusted`。
因子为零或缺失时，调整结果保留为空。
第一式调整混合贡献；第二式将同一贡献因子的平方根用于收益，
是现有实验构造，不能仅凭它将 $\sqrt{\widehat g^{\mathrm{mix},(L)}}$ 称为已估计的收益标准差周期 $\widehat f$。

若先按标准差周期过滤收益 $r^f_{t,i}=r_{t,i}/\widehat f_{t,i}$，
再计算同 Session 的 BV 求和项，恒有：

$$
b^f_{t,i}
=
\frac{b_{t,i}}{\widehat f_{t,i}\widehat f_{t,i-1}}.
$$

特别地，用上面既有的 $r^{\mathrm{adj,mix},(L)}$ 重算 BV 求和项，
除数会是 $\sqrt{\widehat g^{\mathrm{mix},(L)}_{t,i}
\widehat g^{\mathrm{mix},(L)}_{t,i-1}}$，
通常不同于当前直接除以 $\widehat g^{\mathrm{mix},(L)}_{t,i}$。
因此，调整后的混合贡献与“先调整收益再计算 BV”应作为不同对象解释。

#### Cell 10：混合贡献除以 $\widehat g^{\mathrm{mix},(L)}$；既有收益调整构造除以其平方根。

In [ ]:
log_return_matrix = minute_df.pivot(
    index="trading_date",
    columns="minute_slot",
    values="log_return",
).reindex(index=trading_dates, columns=minute_slots)

variance_adjusted_by_lookback = {}
volatility_seasonality_by_lookback = {}
return_adjusted_by_lookback = {}

for lookback in lookback_windows:
    seasonality_matrix = seasonality_by_lookback[lookback]
    positive_seasonality_matrix = seasonality_matrix.where(
        seasonality_matrix > 0
    )

    variance_adjusted_by_lookback[lookback] = (
        variance_contribution_matrix / positive_seasonality_matrix
    )
    volatility_seasonality_by_lookback[lookback] = np.sqrt(
        seasonality_matrix
    )
    return_adjusted_by_lookback[lookback] = (
        log_return_matrix / np.sqrt(positive_seasonality_matrix)
    )

In [45]:
print(return_adjusted_by_lookback.keys())
for key, value in  return_adjusted_by_lookback.items():
    print("回望日历天数 lookback: ", key)
    print(value.columns)
    display(value.tail())

dict_keys([30, 60, 120, 360])
回望日历天数 lookback:  30
Index([09:31:00, 09:32:00, 09:33:00, 09:34:00, 09:35:00, 09:36:00, 09:37:00, 09:38:00, 09:39:00, 09:40:00,
       ...
       14:51:00, 14:52:00, 14:53:00, 14:54:00, 14:55:00, 14:56:00, 14:57:00, 14:58:00, 14:59:00, 15:00:00], dtype='object', name='minute_slot', length=240)


minute_slot,09:31:00,09:32:00,09:33:00,09:34:00,09:35:00,09:36:00,09:37:00,09:38:00,09:39:00,09:40:00,09:41:00,09:42:00,09:43:00,09:44:00,09:45:00,09:46:00,09:47:00,09:48:00,09:49:00,09:50:00,09:51:00,09:52:00,09:53:00,09:54:00,09:55:00,09:56:00,09:57:00,09:58:00,09:59:00,10:00:00,10:01:00,10:02:00,10:03:00,10:04:00,10:05:00,10:06:00,10:07:00,10:08:00,10:09:00,10:10:00,10:11:00,10:12:00,10:13:00,10:14:00,10:15:00,10:16:00,10:17:00,10:18:00,10:19:00,10:20:00,10:21:00,10:22:00,10:23:00,10:24:00,10:25:00,10:26:00,10:27:00,10:28:00,10:29:00,10:30:00,10:31:00,10:32:00,10:33:00,10:34:00,10:35:00,10:36:00,10:37:00,10:38:00,10:39:00,10:40:00,10:41:00,10:42:00,10:43:00,10:44:00,10:45:00,10:46:00,10:47:00,10:48:00,10:49:00,10:50:00,10:51:00,10:52:00,10:53:00,10:54:00,10:55:00,10:56:00,10:57:00,10:58:00,10:59:00,11:00:00,11:01:00,11:02:00,11:03:00,11:04:00,11:05:00,11:06:00,11:07:00,11:08:00,11:09:00,11:10:00,11:11:00,11:12:00,11:13:00,11:14:00,11:15:00,11:16:00,11:17:00,11:18:00,11:19:00,11:20:00,11:21:00,11:22:00,11:23:00,11:24:00,11:25:00,11:26:00,11:27:00,11:28:00,11:29:00,11:30:00,13:01:00,13:02:00,13:03:00,13:04:00,13:05:00,13:06:00,13:07:00,13:08:00,13:09:00,13:10:00,13:11:00,13:12:00,13:13:00,13:14:00,13:15:00,13:16:00,13:17:00,13:18:00,13:19:00,13:20:00,13:21:00,13:22:00,13:23:00,13:24:00,13:25:00,13:26:00,13:27:00,13:28:00,13:29:00,13:30:00,13:31:00,13:32:00,13:33:00,13:34:00,13:35:00,13:36:00,13:37:00,13:38:00,13:39:00,13:40:00,13:41:00,13:42:00,13:43:00,13:44:00,13:45:00,13:46:00,13:47:00,13:48:00,13:49:00,13:50:00,13:51:00,13:52:00,13:53:00,13:54:00,13:55:00,13:56:00,13:57:00,13:58:00,13:59:00,14:00:00,14:01:00,14:02:00,14:03:00,14:04:00,14:05:00,14:06:00,14:07:00,14:08:00,14:09:00,14:10:00,14:11:00,14:12:00,14:13:00,14:14:00,14:15:00,14:16:00,14:17:00,14:18:00,14:19:00,14:20:00,14:21:00,14:22:00,14:23:00,14:24:00,14:25:00,14:26:00,14:27:00,14:28:00,14:29:00,14:30:00,14:31:00,14:32:00,14:33:00,14:34:00,14:35:00,14:36:00,14:37:00,14:38:00,14:39:00,14:40:00,14:41:00,14:42:00,14:43:00,14:44:00,14:45:00,14:46:00,14:47:00,14:48:00,14:49:00,14:50:00,14:51:00,14:52:00,14:53:00,14:54:00,14:55:00,14:56:00,14:57:00,14:58:00,14:59:00,15:00:00
trading_date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2026-08-24,-0.001718,0.000505,0.000395,0.001404,-0.000035,-0.001825,-0.001702,-0.001644,-0.000434,-0.001614,0.000459,0.000637,-0.001046,-0.000540,-0.000698,0.000539,-0.000200,0.000134,0.000592,-0.000864,0.001840,-0.000713,-0.001071,-0.001424,-0.000807,-0.000038,0.000165,-0.001374,-0.001002,-0.000547,0.000077,0.000427,0.001413,-0.000400,-0.000768,0.001400,0.000375,0.000663,-0.000577,0.001867,-0.000515,-0.000845,-0.000812,-0.000154,0.000850,-0.000685,0.000555,0.000116,0.000052,-0.000557,0.000881,0.000430,-0.000769,-0.000851,0.001095,0.001151,-0.000973,-0.000890,-0.000871,-0.000323,-0.000163,0.001338,-0.000235,0.001086,-0.001826,0.000170,-0.000165,-0.000385,-0.000241,-0.000177,-0.000314,-0.000690,-0.001376,-0.001501,-0.000387,0.000643,-0.000505,-0.000250,0.000780,0.000000,-0.000178,-0.001478,0.000803,-0.000443,-0.000287,0.000689,-0.001328,0.000204,-0.000224,0.000104,-0.000492,0.000131,-0.000272,-0.000413,0.000355,-0.000322,-0.000764,0.002074,0.000501,-0.000027,-0.000082,-0.000741,0.000332,0.001818,0.000511,0.001397,-0.001103,0.000000,0.001190,-0.000442,-0.001721,-0.001529,-0.002473,0.000608,0.000566,0.001638,-0.002188,0.001116,0.000173,-0.000669,-0.001070,0.000291,-0.001300,-0.001302,-0.000385,0.002910,0.001209,0.000845,0.000134,-0.000664,-0.001312,-0.000154,0.000483,0.000078,-0.000864,-0.000160,-0.000190,-0.000048,-0.000056,-0.000162,-0.000218,-0.000781,0.000143,0.001543,0.000392,-0.001040,0.000030,-0.000541,-0.000375,0.000274,-0.000352,-0.000747,-0.000801,-0.000879,-0.000206,-0.000321,-0.000669,-0.000393,0.000896,0.000000,0.000154,0.000471,-0.000536,

回望日历天数 lookback:  60
Index([09:31:00, 09:32:00, 09:33:00, 09:34:00, 09:35:00, 09:36:00, 09:37:00, 09:38:00, 09:39:00, 09:40:00,
       ...
       14:51:00, 14:52:00, 14:53:00, 14:54:00, 14:55:00, 14:56:00, 14:57:00, 14:58:00, 14:59:00, 15:00:00], dtype='object', name='minute_slot', length=240)


minute_slot,09:31:00,09:32:00,09:33:00,09:34:00,09:35:00,09:36:00,09:37:00,09:38:00,09:39:00,09:40:00,09:41:00,09:42:00,09:43:00,09:44:00,09:45:00,09:46:00,09:47:00,09:48:00,09:49:00,09:50:00,09:51:00,09:52:00,09:53:00,09:54:00,09:55:00,09:56:00,09:57:00,09:58:00,09:59:00,10:00:00,10:01:00,10:02:00,10:03:00,10:04:00,10:05:00,10:06:00,10:07:00,10:08:00,10:09:00,10:10:00,10:11:00,10:12:00,10:13:00,10:14:00,10:15:00,10:16:00,10:17:00,10:18:00,10:19:00,10:20:00,10:21:00,10:22:00,10:23:00,10:24:00,10:25:00,10:26:00,10:27:00,10:28:00,10:29:00,10:30:00,10:31:00,10:32:00,10:33:00,10:34:00,10:35:00,10:36:00,10:37:00,10:38:00,10:39:00,10:40:00,10:41:00,10:42:00,10:43:00,10:44:00,10:45:00,10:46:00,10:47:00,10:48:00,10:49:00,10:50:00,10:51:00,10:52:00,10:53:00,10:54:00,10:55:00,10:56:00,10:57:00,10:58:00,10:59:00,11:00:00,11:01:00,11:02:00,11:03:00,11:04:00,11:05:00,11:06:00,11:07:00,11:08:00,11:09:00,11:10:00,11:11:00,11:12:00,11:13:00,11:14:00,11:15:00,11:16:00,11:17:00,11:18:00,11:19:00,11:20:00,11:21:00,11:22:00,11:23:00,11:24:00,11:25:00,11:26:00,11:27:00,11:28:00,11:29:00,11:30:00,13:01:00,13:02:00,13:03:00,13:04:00,13:05:00,13:06:00,13:07:00,13:08:00,13:09:00,13:10:00,13:11:00,13:12:00,13:13:00,13:14:00,13:15:00,13:16:00,13:17:00,13:18:00,13:19:00,13:20:00,13:21:00,13:22:00,13:23:00,13:24:00,13:25:00,13:26:00,13:27:00,13:28:00,13:29:00,13:30:00,13:31:00,13:32:00,13:33:00,13:34:00,13:35:00,13:36:00,13:37:00,13:38:00,13:39:00,13:40:00,13:41:00,13:42:00,13:43:00,13:44:00,13:45:00,13:46:00,13:47:00,13:48:00,13:49:00,13:50:00,13:51:00,13:52:00,13:53:00,13:54:00,13:55:00,13:56:00,13:57:00,13:58:00,13:59:00,14:00:00,14:01:00,14:02:00,14:03:00,14:04:00,14:05:00,14:06:00,14:07:00,14:08:00,14:09:00,14:10:00,14:11:00,14:12:00,14:13:00,14:14:00,14:15:00,14:16:00,14:17:00,14:18:00,14:19:00,14:20:00,14:21:00,14:22:00,14:23:00,14:24:00,14:25:00,14:26:00,14:27:00,14:28:00,14:29:00,14:30:00,14:31:00,14:32:00,14:33:00,14:34:00,14:35:00,14:36:00,14:37:00,14:38:00,14:39:00,14:40:00,14:41:00,14:42:00,14:43:00,14:44:00,14:45:00,14:46:00,14:47:00,14:48:00,14:49:00,14:50:00,14:51:00,14:52:00,14:53:00,14:54:00,14:55:00,14:56:00,14:57:00,14:58:00,14:59:00,15:00:00
trading_date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2026-08-24,-0.001577,0.000418,0.000398,0.001406,-0.000032,-0.002076,-0.001817,-0.001571,-0.000436,-0.001425,0.000408,0.000651,-0.001239,-0.000540,-0.000728,0.000535,-0.000194,0.000119,0.000588,-0.000851,0.001732,-0.000648,-0.001233,-0.001414,-0.000673,-0.000042,0.000178,-0.001362,-0.000814,-0.000506,0.000075,0.000395,0.000991,-0.000353,-0.000621,0.001411,0.000365,0.000708,-0.000586,0.001794,-0.000524,-0.000867,-0.000848,-0.000139,0.000845,-0.000661,0.000594,0.000118,0.000049,-0.000538,0.000857,0.000452,-0.000908,-0.000890,0.001175,0.001098,-0.000843,-0.000868,-0.000781,-0.000336,-0.000180,0.001681,-0.000233,0.000764,-0.001827,0.000180,-0.000159,-0.000321,-0.000232,-0.000174,-0.000251,-0.000632,-0.001169,-0.001348,-0.000372,0.000648,-0.000695,-0.000272,0.000797,0.000000,-0.000173,-0.001322,0.000781,-0.000561,-0.000275,0.000657,-0.001258,0.000142,-0.000210,0.000100,-0.000537,0.000139,-0.000276,-0.000401,0.000305,-0.000330,-0.000780,0.002071,0.000521,-0.000027,-0.000086,-0.000772,0.000329,0.002285,0.000521,0.001388,-0.001068,0.000000,0.001311,-0.000634,-0.001937,-0.001365,-0.001921,0.000649,0.000535,0.001458,-0.001899,0.001320,0.000202,-0.000707,-0.000924,0.000237,-0.001183,-0.001293,-0.000383,0.002891,0.001027,0.000870,0.000153,-0.000727,-0.001226,-0.000162,0.000506,0.000070,-0.000817,-0.000179,-0.000257,-0.000057,-0.000061,-0.000138,-0.000195,-0.000910,0.000170,0.001704,0.000415,-0.000972,0.000030,-0.000555,-0.000366,0.000273,-0.000301,-0.000677,-0.000732,-0.000862,-0.000226,-0.000259,-0.000697,-0.000427,0.000996,0.000000,0.000137,0.000395,-0.000461,

回望日历天数 lookback:  120
Index([09:31:00, 09:32:00, 09:33:00, 09:34:00, 09:35:00, 09:36:00, 09:37:00, 09:38:00, 09:39:00, 09:40:00,
       ...
       14:51:00, 14:52:00, 14:53:00, 14:54:00, 14:55:00, 14:56:00, 14:57:00, 14:58:00, 14:59:00, 15:00:00], dtype='object', name='minute_slot', length=240)


minute_slot,09:31:00,09:32:00,09:33:00,09:34:00,09:35:00,09:36:00,09:37:00,09:38:00,09:39:00,09:40:00,09:41:00,09:42:00,09:43:00,09:44:00,09:45:00,09:46:00,09:47:00,09:48:00,09:49:00,09:50:00,09:51:00,09:52:00,09:53:00,09:54:00,09:55:00,09:56:00,09:57:00,09:58:00,09:59:00,10:00:00,10:01:00,10:02:00,10:03:00,10:04:00,10:05:00,10:06:00,10:07:00,10:08:00,10:09:00,10:10:00,10:11:00,10:12:00,10:13:00,10:14:00,10:15:00,10:16:00,10:17:00,10:18:00,10:19:00,10:20:00,10:21:00,10:22:00,10:23:00,10:24:00,10:25:00,10:26:00,10:27:00,10:28:00,10:29:00,10:30:00,10:31:00,10:32:00,10:33:00,10:34:00,10:35:00,10:36:00,10:37:00,10:38:00,10:39:00,10:40:00,10:41:00,10:42:00,10:43:00,10:44:00,10:45:00,10:46:00,10:47:00,10:48:00,10:49:00,10:50:00,10:51:00,10:52:00,10:53:00,10:54:00,10:55:00,10:56:00,10:57:00,10:58:00,10:59:00,11:00:00,11:01:00,11:02:00,11:03:00,11:04:00,11:05:00,11:06:00,11:07:00,11:08:00,11:09:00,11:10:00,11:11:00,11:12:00,11:13:00,11:14:00,11:15:00,11:16:00,11:17:00,11:18:00,11:19:00,11:20:00,11:21:00,11:22:00,11:23:00,11:24:00,11:25:00,11:26:00,11:27:00,11:28:00,11:29:00,11:30:00,13:01:00,13:02:00,13:03:00,13:04:00,13:05:00,13:06:00,13:07:00,13:08:00,13:09:00,13:10:00,13:11:00,13:12:00,13:13:00,13:14:00,13:15:00,13:16:00,13:17:00,13:18:00,13:19:00,13:20:00,13:21:00,13:22:00,13:23:00,13:24:00,13:25:00,13:26:00,13:27:00,13:28:00,13:29:00,13:30:00,13:31:00,13:32:00,13:33:00,13:34:00,13:35:00,13:36:00,13:37:00,13:38:00,13:39:00,13:40:00,13:41:00,13:42:00,13:43:00,13:44:00,13:45:00,13:46:00,13:47:00,13:48:00,13:49:00,13:50:00,13:51:00,13:52:00,13:53:00,13:54:00,13:55:00,13:56:00,13:57:00,13:58:00,13:59:00,14:00:00,14:01:00,14:02:00,14:03:00,14:04:00,14:05:00,14:06:00,14:07:00,14:08:00,14:09:00,14:10:00,14:11:00,14:12:00,14:13:00,14:14:00,14:15:00,14:16:00,14:17:00,14:18:00,14:19:00,14:20:00,14:21:00,14:22:00,14:23:00,14:24:00,14:25:00,14:26:00,14:27:00,14:28:00,14:29:00,14:30:00,14:31:00,14:32:00,14:33:00,14:34:00,14:35:00,14:36:00,14:37:00,14:38:00,14:39:00,14:40:00,14:41:00,14:42:00,14:43:00,14:44:00,14:45:00,14:46:00,14:47:00,14:48:00,14:49:00,14:50:00,14:51:00,14:52:00,14:53:00,14:54:00,14:55:00,14:56:00,14:57:00,14:58:00,14:59:00,15:00:00
trading_date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2026-08-24,-0.001425,0.000354,0.000367,0.001326,-0.000029,-0.001708,-0.001710,-0.001456,-0.000436,-0.001336,0.000376,0.000632,-0.001130,-0.000531,-0.000770,0.000552,-0.000203,0.000131,0.000572,-0.000794,0.001619,-0.000564,-0.001066,-0.001301,-0.000707,-0.000043,0.000182,-0.001343,-0.000787,-0.000470,0.000069,0.000401,0.000983,-0.000276,-0.000565,0.001380,0.000362,0.000718,-0.000520,0.001743,-0.000506,-0.000860,-0.000841,-0.000138,0.000894,-0.000666,0.000587,0.000120,0.000049,-0.000555,0.000929,0.000464,-0.000948,-0.000964,0.001171,0.001031,-0.000847,-0.000990,-0.000862,-0.000345,-0.000180,0.001659,-0.000229,0.000786,-0.001702,0.000160,-0.000152,-0.000326,-0.000219,-0.000133,-0.000264,-0.000651,-0.001202,-0.001516,-0.000399,0.000694,-0.000693,-0.000268,0.000833,0.000000,-0.000185,-0.001457,0.000817,-0.000620,-0.000272,0.000570,-0.001177,0.000147,-0.000230,0.000102,-0.000505,0.000121,-0.000266,-0.000391,0.000316,-0.000329,-0.000731,0.002061,0.000569,-0.000029,-0.000099,-0.000829,0.000341,0.002277,0.000556,0.001513,-0.001129,0.000000,0.001367,-0.000637,-0.001860,-0.001212,-0.001723,0.000644,0.000509,0.001501,-0.001905,0.001296,0.000197,-0.000703,-0.000829,0.000198,-0.000952,-0.001353,-0.000412,0.002614,0.000840,0.000863,0.000187,-0.000682,-0.001115,-0.000152,0.000542,0.000075,-0.000852,-0.000181,-0.000286,-0.000059,-0.000065,-0.000147,-0.000239,-0.000923,0.000165,0.001690,0.000463,-0.001066,0.000032,-0.000644,-0.000399,0.000295,-0.000341,-0.000724,-0.000736,-0.000897,-0.000232,-0.000271,-0.000642,-0.000450,0.001179,0.000000,0.000147,0.000410,-0.000501,

回望日历天数 lookback:  360
Index([09:31:00, 09:32:00, 09:33:00, 09:34:00, 09:35:00, 09:36:00, 09:37:00, 09:38:00, 09:39:00, 09:40:00,
       ...
       14:51:00, 14:52:00, 14:53:00, 14:54:00, 14:55:00, 14:56:00, 14:57:00, 14:58:00, 14:59:00, 15:00:00], dtype='object', name='minute_slot', length=240)


minute_slot,09:31:00,09:32:00,09:33:00,09:34:00,09:35:00,09:36:00,09:37:00,09:38:00,09:39:00,09:40:00,09:41:00,09:42:00,09:43:00,09:44:00,09:45:00,09:46:00,09:47:00,09:48:00,09:49:00,09:50:00,09:51:00,09:52:00,09:53:00,09:54:00,09:55:00,09:56:00,09:57:00,09:58:00,09:59:00,10:00:00,10:01:00,10:02:00,10:03:00,10:04:00,10:05:00,10:06:00,10:07:00,10:08:00,10:09:00,10:10:00,10:11:00,10:12:00,10:13:00,10:14:00,10:15:00,10:16:00,10:17:00,10:18:00,10:19:00,10:20:00,10:21:00,10:22:00,10:23:00,10:24:00,10:25:00,10:26:00,10:27:00,10:28:00,10:29:00,10:30:00,10:31:00,10:32:00,10:33:00,10:34:00,10:35:00,10:36:00,10:37:00,10:38:00,10:39:00,10:40:00,10:41:00,10:42:00,10:43:00,10:44:00,10:45:00,10:46:00,10:47:00,10:48:00,10:49:00,10:50:00,10:51:00,10:52:00,10:53:00,10:54:00,10:55:00,10:56:00,10:57:00,10:58:00,10:59:00,11:00:00,11:01:00,11:02:00,11:03:00,11:04:00,11:05:00,11:06:00,11:07:00,11:08:00,11:09:00,11:10:00,11:11:00,11:12:00,11:13:00,11:14:00,11:15:00,11:16:00,11:17:00,11:18:00,11:19:00,11:20:00,11:21:00,11:22:00,11:23:00,11:24:00,11:25:00,11:26:00,11:27:00,11:28:00,11:29:00,11:30:00,13:01:00,13:02:00,13:03:00,13:04:00,13:05:00,13:06:00,13:07:00,13:08:00,13:09:00,13:10:00,13:11:00,13:12:00,13:13:00,13:14:00,13:15:00,13:16:00,13:17:00,13:18:00,13:19:00,13:20:00,13:21:00,13:22:00,13:23:00,13:24:00,13:25:00,13:26:00,13:27:00,13:28:00,13:29:00,13:30:00,13:31:00,13:32:00,13:33:00,13:34:00,13:35:00,13:36:00,13:37:00,13:38:00,13:39:00,13:40:00,13:41:00,13:42:00,13:43:00,13:44:00,13:45:00,13:46:00,13:47:00,13:48:00,13:49:00,13:50:00,13:51:00,13:52:00,13:53:00,13:54:00,13:55:00,13:56:00,13:57:00,13:58:00,13:59:00,14:00:00,14:01:00,14:02:00,14:03:00,14:04:00,14:05:00,14:06:00,14:07:00,14:08:00,14:09:00,14:10:00,14:11:00,14:12:00,14:13:00,14:14:00,14:15:00,14:16:00,14:17:00,14:18:00,14:19:00,14:20:00,14:21:00,14:22:00,14:23:00,14:24:00,14:25:00,14:26:00,14:27:00,14:28:00,14:29:00,14:30:00,14:31:00,14:32:00,14:33:00,14:34:00,14:35:00,14:36:00,14:37:00,14:38:00,14:39:00,14:40:00,14:41:00,14:42:00,14:43:00,14:44:00,14:45:00,14:46:00,14:47:00,14:48:00,14:49:00,14:50:00,14:51:00,14:52:00,14:53:00,14:54:00,14:55:00,14:56:00,14:57:00,14:58:00,14:59:00,15:00:00
trading_date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2026-08-24,-0.001010,0.000273,0.000302,0.001289,-0.000028,-0.001486,-0.001523,-0.001569,-0.000498,-0.001364,0.000398,0.000604,-0.001150,-0.000564,-0.000780,0.000614,-0.000243,0.000146,0.000589,-0.000828,0.001488,-0.000598,-0.001086,-0.001317,-0.000686,-0.000045,0.000197,-0.001342,-0.000773,-0.000473,0.000063,0.000416,0.000991,-0.000274,-0.000578,0.001378,0.000382,0.000808,-0.000537,0.001752,-0.000565,-0.000911,-0.000825,-0.000154,0.000895,-0.000650,0.000637,0.000125,0.000049,-0.000536,0.000889,0.000478,-0.001050,-0.000974,0.001118,0.001009,-0.000800,-0.001040,-0.000993,-0.000322,-0.000170,0.001468,-0.000224,0.000838,-0.001671,0.000170,-0.000148,-0.000331,-0.000244,-0.000155,-0.000288,-0.000751,-0.001260,-0.001630,-0.000438,0.000843,-0.000846,-0.000312,0.000930,0.000000,-0.000192,-0.001359,0.000896,-0.000597,-0.000282,0.000654,-0.001212,0.000153,-0.000235,0.000095,-0.000499,0.000136,-0.000282,-0.000398,0.000300,-0.000327,-0.000797,0.002183,0.000611,-0.000031,-0.000098,-0.000789,0.000332,0.002068,0.000578,0.001548,-0.001118,0.000000,0.001344,-0.000582,-0.001783,-0.001135,-0.001977,0.000562,0.000519,0.001463,-0.001788,0.001276,0.000200,-0.000649,-0.000889,0.000218,-0.001008,-0.001246,-0.000425,0.002852,0.000920,0.000903,0.000190,-0.000763,-0.001203,-0.000168,0.000640,0.000074,-0.000789,-0.000176,-0.000268,-0.000066,-0.000067,-0.000136,-0.000233,-0.000975,0.000174,0.001884,0.000491,-0.001031,0.000037,-0.000759,-0.000419,0.000313,-0.000392,-0.000819,-0.000778,-0.000873,-0.000241,-0.000313,-0.000704,-0.000543,0.001278,0.000000,0.000171,0.000451,-0.000544,

#### Cell 11：整理各窗口的分钟结果，保留 RV／BPV 类型标记。

In [43]:
output_index = pd.MultiIndex.from_product(
    [trading_dates, minute_slots],
    names=["trading_date", "minute_slot"],
)

base_output_df = minute_df.set_index(
    ["trading_date", "minute_slot"]
)[[
    "contract_code", "session_number", "bar_at",
    "log_return", "bpv_contribution", "variance_contribution",
]].reindex(output_index)

base_output_df["contribution_method"] = np.where(
    output_index.get_level_values("minute_slot").isin([
        dt.time(9, 31), dt.time(13, 1),
    ]),
    "rv_open",
    "bpv",
)

output_by_lookback = {}

for lookback in lookback_windows:
    output_df = base_output_df.copy()
    output_df["lookback_trading_days"] = lookback

    output_df["variance_seasonality"] = (
        seasonality_by_lookback[lookback].to_numpy().ravel()
    )
    output_df["variance_adjusted"] = (
        variance_adjusted_by_lookback[lookback].to_numpy().ravel()
    )
    output_df["volatility_seasonality"] = (
        volatility_seasonality_by_lookback[lookback].to_numpy().ravel()
    )
    output_df["return_adjusted"] = (
        return_adjusted_by_lookback[lookback].to_numpy().ravel()
    )

    output_by_lookback[lookback] = output_df.reset_index()

In [46]:
print(output_by_lookback.keys())
for key, value in  output_by_lookback.items():
    print("回望日历天数 lookback: ", key)
    print(value.columns)
    display(value.tail())

dict_keys([30, 60, 120, 360])
回望日历天数 lookback:  30
Index(['trading_date', 'minute_slot', 'contract_code', 'session_number', 'bar_at', 'log_return', 'bpv_contribution', 'variance_contribution', 'contribution_method', 'lookback_trading_days', 'variance_seasonality', 'variance_adjusted', 'volatility_seasonality', 'return_adjusted'], dtype='str')


,trading_date,minute_slot,contract_code,session_number,bar_at,log_return,bpv_contribution,variance_contribution,contribution_method,lookback_trading_days,variance_seasonality,variance_adjusted,volatility_seasonality,return_adjusted
238795,2026-08-28,14:56:00,IM2609.CCFX,2,2026-08-28 14:56:00+08:00,0.000392,3.212797e-08,3.212797e-08,bpv,30,0.182146,1.763859e-07,0.426786,0.000918
238796,2026-08-28,14:57:00,IM2609.CCFX,2,2026-08-28 14:57:00+08:00,-0.000078,4.817623e-08,4.817623e-08,bpv,30,0.146668,3.284703e-07,0.382973,-0.000204
238797,2026-08-28,14:58:00,IM2609.CCFX,2,2026-08-28 14:58:00+08:00,0.000470,5.779111e-08,5.779111e-08,bpv,30,0.256475,2.253281e-07,0.506434,0.000928
238798,2026-08-28,14:59:00,IM2609.CCFX,2,2026-08-28 14:59:00+08:00,-0.000026,1.925566e-08,1.925566e-08,bpv,30,0.405001,4.754477e-08,0.636397,-0.000041
238799,2026-08-28,15:00:00,IM2609.CCFX,2,2026-08-28 15:00:00+08:00,0.000261,1.069396e-08,1.069396e-08,bpv,30,0.302907,3.530447e-08,0.550370,0.000474


回望日历天数 lookback:  60
Index(['trading_date', 'minute_slot', 'contract_code', 'session_number', 'bar_at', 'log_return', 'bpv_contribution', 'variance_contribution', 'contribution_method', 'lookback_trading_days', 'variance_seasonality', 'variance_adjusted', 'volatility_seasonality', 'return_adjusted'], dtype='str')


,trading_date,minute_slot,contract_code,session_number,bar_at,log_return,bpv_contribution,variance_contribution,contribution_method,lookback_trading_days,variance_seasonality,variance_adjusted,volatility_seasonality,return_adjusted
238795,2026-08-28,14:56:00,IM2609.CCFX,2,2026-08-28 14:56:00+08:00,0.000392,3.212797e-08,3.212797e-08,bpv,60,0.186212,1.725343e-07,0.431523,0.000908
238796,2026-08-28,14:57:00,IM2609.CCFX,2,2026-08-28 14:57:00+08:00,-0.000078,4.817623e-08,4.817623e-08,bpv,60,0.180255,2.672666e-07,0.424565,-0.000184
238797,2026-08-28,14:58:00,IM2609.CCFX,2,2026-08-28 14:58:00+08:00,0.000470,5.779111e-08,5.779111e-08,bpv,60,0.289930,1.993279e-07,0.538451,0.000872
238798,2026-08-28,14:59:00,IM2609.CCFX,2,2026-08-28 14:59:00+08:00,-0.000026,1.925566e-08,1.925566e-08,bpv,60,0.332926,5.783770e-08,0.576997,-0.000045
238799,2026-08-28,15:00:00,IM2609.CCFX,2,2026-08-28 15:00:00+08:00,0.000261,1.069396e-08,1.069396e-08,bpv,60,0.285894,3.740538e-08,0.534690,0.000488


回望日历天数 lookback:  120
Index(['trading_date', 'minute_slot', 'contract_code', 'session_number', 'bar_at', 'log_return', 'bpv_contribution', 'variance_contribution', 'contribution_method', 'lookback_trading_days', 'variance_seasonality', 'variance_adjusted', 'volatility_seasonality', 'return_adjusted'], dtype='str')


,trading_date,minute_slot,contract_code,session_number,bar_at,log_return,bpv_contribution,variance_contribution,contribution_method,lookback_trading_days,variance_seasonality,variance_adjusted,volatility_seasonality,return_adjusted
238795,2026-08-28,14:56:00,IM2609.CCFX,2,2026-08-28 14:56:00+08:00,0.000392,3.212797e-08,3.212797e-08,bpv,120,0.187851,1.710293e-07,0.433417,0.000904
238796,2026-08-28,14:57:00,IM2609.CCFX,2,2026-08-28 14:57:00+08:00,-0.000078,4.817623e-08,4.817623e-08,bpv,120,0.211873,2.273824e-07,0.460297,-0.000170
238797,2026-08-28,14:58:00,IM2609.CCFX,2,2026-08-28 14:58:00+08:00,0.000470,5.779111e-08,5.779111e-08,bpv,120,0.337923,1.710183e-07,0.581312,0.000808
238798,2026-08-28,14:59:00,IM2609.CCFX,2,2026-08-28 14:59:00+08:00,-0.000026,1.925566e-08,1.925566e-08,bpv,120,0.486242,3.960097e-08,0.697311,-0.000037
238799,2026-08-28,15:00:00,IM2609.CCFX,2,2026-08-28 15:00:00+08:00,0.000261,1.069396e-08,1.069396e-08,bpv,120,0.316751,3.376140e-08,0.562806,0.000464


回望日历天数 lookback:  360
Index(['trading_date', 'minute_slot', 'contract_code', 'session_number', 'bar_at', 'log_return', 'bpv_contribution', 'variance_contribution', 'contribution_method', 'lookback_trading_days', 'variance_seasonality', 'variance_adjusted', 'volatility_seasonality', 'return_adjusted'], dtype='str')


,trading_date,minute_slot,contract_code,session_number,bar_at,log_return,bpv_contribution,variance_contribution,contribution_method,lookback_trading_days,variance_seasonality,variance_adjusted,volatility_seasonality,return_adjusted
238795,2026-08-28,14:56:00,IM2609.CCFX,2,2026-08-28 14:56:00+08:00,0.000392,3.212797e-08,3.212797e-08,bpv,360,0.288427,1.113901e-07,0.537054,0.000729
238796,2026-08-28,14:57:00,IM2609.CCFX,2,2026-08-28 14:57:00+08:00,-0.000078,4.817623e-08,4.817623e-08,bpv,360,0.321600,1.498017e-07,0.567098,-0.000138
238797,2026-08-28,14:58:00,IM2609.CCFX,2,2026-08-28 14:58:00+08:00,0.000470,5.779111e-08,5.779111e-08,bpv,360,0.389821,1.482505e-07,0.624356,0.000752
238798,2026-08-28,14:59:00,IM2609.CCFX,2,2026-08-28 14:59:00+08:00,-0.000026,1.925566e-08,1.925566e-08,bpv,360,0.415086,4.638954e-08,0.644272,-0.000041
238799,2026-08-28,15:00:00,IM2609.CCFX,2,2026-08-28 15:00:00+08:00,0.000261,1.069396e-08,1.069396e-08,bpv,360,0.372116,2.873824e-08,0.610013,0.000428


#### Cell 12：定义输出 Schema，注明“首分钟 RV 项＋其余分钟 BV 求和项”的混合口径。

现有字段 `variance_seasonality` 表示 $\widehat g^{\mathrm{mix},(L)}$，
`volatility_seasonality` 表示其平方根；Schema 中保留的
`seasonality_scale=variance` 是旧标签，不构成该因子等于收益方差周期 $f^2$ 的证明。
字段名称、Schema 与既有输出沿用当前实现，统计含义以上述定义为准。

In [48]:
output_schema = pa.schema([
    pa.field("trading_date", pa.date32(), nullable=False),
    pa.field("minute_slot", pa.time64("us"), nullable=False),
    pa.field("lookback_trading_days", pa.int16(), nullable=False),
    pa.field("contract_code", pa.string()),
    pa.field("session_number", pa.int8()),
    pa.field("bar_at", pa.timestamp("us", tz="Asia/Shanghai")),
    pa.field("contribution_method", pa.string(), nullable=False),

    *[
        pa.field(column, pa.float64())
        for column in [
            "log_return",
            "bpv_contribution",
            "variance_contribution",
            "variance_seasonality",
            "variance_adjusted",
            "volatility_seasonality",
            "return_adjusted",
        ]
    ],
], metadata={
    b"primary_key": b"trading_date,minute_slot",
    b"contribution_method": b"RV at 09:31 and 13:01; BPV elsewhere within session",
    b"seasonality_scale": b"variance",
    b"estimation": b"daily normalization; shift(1); rolling median; mean normalization",
    b"source_file": source_path.name.encode("utf-8"),
})

In [49]:
for lookback in lookback_windows:
    window_schema = output_schema.with_metadata({
        **output_schema.metadata,
        b"lookback_trading_days": str(lookback).encode("utf-8"),
    })

    output_table = pa.Table.from_pandas(
        output_by_lookback[lookback][window_schema.names],
        schema=window_schema,
        preserve_index=False,
    )

    output_path = (
        research_dir / f"im_bpv_seasonality_1m_{lookback}td.parquet"
    )
    temporary_output_path = output_path.with_suffix(".parquet.tmp")

    pq.write_table(output_table, temporary_output_path, compression="zstd")
    temporary_output_path.replace(output_path)

    print(f"已保存：{output_path.name}")

已保存：im_bpv_seasonality_1m_30td.parquet
已保存：im_bpv_seasonality_1m_60td.parquet
已保存：im_bpv_seasonality_1m_120td.parquet
已保存：im_bpv_seasonality_1m_360td.parquet
